# Kaggriculture 

A guarded hybrid of two complete, immutable routes.


## Method

The selector chooses one complete route once at the beginning of each episode. It never mixes routes mid-game and never edits an action. High MELON or STRAWBERRY prices select the verified premium-control route; otherwise the v22 price-impact route is used. If the observation is incomplete, 22 is the fallback.


In [ ]:
V22_SOURCE = r'''"""v22 price-impact route agent for Kaggriculture.

The production route is a real fit-only trajectory.  Runtime adaptation is
limited to identity-free weed recovery and in-place ranking of existing SELL
slots by the official 1.32.4 price-impact curve.  Ordinary turns never create,
delete, or resize a market order.
"""
import base64
import copy
import json
import math
import zlib


_ACTIONS = json.loads(zlib.decompress(base64.b85decode(
    (
    'c-rk<U2hyoa{MoR<^$)06zMmvG-nCN6$MJV!FfR}7VsGcjPt|VZ^r$1Yei0XPiJIgWLEVowYs+fIn!O0Rb8DK85#N0|DFBEFTei%'
    'Z@-@X%TH$?Za;oJdptk;&tLxg-~Z>!AHID2`!B!#*Wdp4%jciY-oAU-efh8U;fK$E{rUF&yB}}w&d$$1zTNFUoSm=DKVIMOCx8C9'
    '+r9bn$Nk&g?WeQzSF=C=xVyW5e|ElFKR*8B{AkqgUjO;?hso83@&9zT-+lb_bv*CyA3nYP`ssO+liy8;_w<9~iT^f;4-'
    'fZmKEM7n4$lnZhtKcs-'
    'u(RK>YqM;wZUW*<IUMHh6|71n~q~X>$}_6yXQ&kH#7evcX+nl<kIsg!dtjsBDW&88&>ekgx`<$KQ`gBEuI##(SC>fJngryd*b%f?'
    '&0{3fBJ1tPDl0p?UXsk>yDE=+~D=?v+<t4)X8|`q=q{U-'
    '?3XioPk{vU@N;JW<TSrbaVsJdUiv^W;|RkX})16G?<TWwP7ddTWx+_(Q4yP=wdAVpw1^8S#ADK60J7*lWsP1t4`JebMP%<{xx~H3'
    'dRBk@ogkLkYp<6Lnjm25025gjeBORZsR`waQe$WpCyii2mPFl>uwESNL|nPP0t5t(>3Nt>+dy=f_=?34ldOnVluni_J!#&j@S2hc'
    'e~f0e*V+$;nTajcmH<$@~T|%<NfFMW$HhyHxKt8mVKH&?(TjI-'
    '6lgGBe+F2M0f(N8n5?ao;YUs^3KWZ+g>*TF>P`;sTf0HbvY^!M~?HAo?d2j*7fVn&$pxNp%pM57BuPja5$D)J%$0wI1u3fTA!}r-'
    'qxt26K0Ltb=pn#kByKx9CHvsY=q3MNkCUx`(D!qVas<qZ*Y<<G;uc~>U8h969A_>e0ciya<~2tX3byZN-VsX4alwkm?kKM_Am9$e'
    'Xsv5U2W#yZZrPvR`qYWqr1h~G{v)0l48#nQ&30dK!ID#Z!bhjDOWXl%QkZzWsz#${v0LkZ7UQ2F}HH|zsfDG5$%jXP7)rpRVSX@u'
    '`tEVn~c5I>o+7e4I%g*u$y?lmZ)&k@ui)3ktGIX(39U`3@^?oAhG$~+XNiC|1?Uktk#Ph!Z)25T<WrNuF&(7owt7%5Bkz0p7rsdr'
    'vX}TADZ&G7ILBorc09(O{h){QvqIe3?VLOJv+&fCZsq>32>ZI#~M&_!5x(17Gb)Z#Q^f*e{XLe{-Qd<8v!YOcK-'
    'R)byCeRJa`Wj&(1e@m3L@|KaNGAv{}<ke~$xZ%orqhLB5o^%wR4l$`inFW~BN0XY#G%pQfL{rK3R;C>zlTaVAS(W)TYJ;eOLaw=;'
    'oHUuFdeqtHvx$<Lm(;?Wae4LPn^Y`f3{HxmOp;+fJh{4cj9h6#Wx&*v1L(lA?}oHO11&T_^xPb0F2SIng|uZ>^MfN@rXoSFxv;4l'
    '?{#MG<c=_09tp(RQUGmIMC*f}-MG#OI(i3j?n24>S*;lPqMuNcs+hX{x3Vqy&kV{*bC0@ZRfS3~CS<tHtn{bhS-'
    '4}X2tH+0|hkJwKZr*}(n*;mA(L3tkou?bb|!^Vva)RBxvkdw}gpj@EXm7S5JVc8Wt-L&J?-'
    'Xt<7cG?U?($A1hE(xqWj2To!A<iX27e5Ls4p8A5HL<s{{)-'
    ')JGF;1`4)%gdgs#VR&E~MSCM&mZ9XPqRVi!Buh%{U^%iKM=3}f8;`@63d^Ekx4n*6=ue%;-'
    '^f0Hlvj{~=PJwNt$kh>tdi8r&v&&N*>xBDM<4-bDmJHL+K$g~XJ&;Bfzwwbe##`D*gI7o^CcwsEQ-gq=-'
    '?x9$EyfXOTVIV^WWcHn<t?gBv`@mNC+F^DdE<Vr9p)AB?kGHM?D0e~c>b|sD;J^S&fTYP_iG_LO$k0avVVikAgHEFqBN<OV<M?C?'
    'Hc|^W+ibdF9fuF|1Zt3lX>iKrr3DpA=m-?R^(|rfCYID%4IRYUg-'
    'i^NVIrfOg{=w3EUfD#VRsahG43j(Povi>1mlUp0tcSFRHx<}M-'
    '(3=9j%9UerVBo%+P79gpg>mcSXKe&wXdd)<i|CnWIToLY*P0wbJj#sFiZ)bs`SiRf(y_h0vza$~F;BO^<W<f)|Cz8u0nmMf}DXrL'
    '}CCKOT7UlMn@5KZ==C=T5^MfIr$cvYSUeKtOjIhE2a!;m2i`*2t&L-'
    'w?aGBLl|PWsqcHL2zT&)s7}`U&L`ykpUJ<BE>Mzx_~^l(1LUCGo&X)hL$r&7ulCHk8m0C;SRXdK856~iF>o-'
    '?lxvQb+IWxv&{pK7*7uI%8@|}tB$mj9da+hs!u#6pN*z5D}`YiAC9o_oJR+pn69V*&XkstEMN<=Y%FCF7s#y|Ay7C)!J^v#wWXoB'
    'CLdztRz@HvDC~0ud7YBzXJuG0g!Jz!)mwJW;4q506jC+k!IC#-V+o<>xs({cuDdhg&7ZCf!>1qL-TfIbdEhzSZ$Kohrfk=ok)rUF'
    'q$0@x9gF4Dh0ibaO(|&QqTbV>^_7#4T;+q=wHnN>nu6J<4>baq6;x=6Vds}iy&Gp*#W=pog{;No;jBg}0Q&olgk&0QKID+#(_&^P'
    'XEI)?wp75K(qkn$-'
    '~x5oE*n`$KT+dHB?Z#94aH;ofTA%uRIcEY^wvCzB#T<{vA~Y7OBkjy#XNJ~<Ixh9nFZF95iAEA?m|@6;^S$m_uA|)qGxcIdl(zxb'
    'AnH543b5JuZ7(dZnDr>ON|ivw}_+es-'
    'Z8jdJ#M(EmP;UY|>!qH3;JOFE@r6v2VMa;72brwMfI9Y7c*(R%w>;&&G&KG|jReN|d)jf4tNKJ`8ZLGm)R=UgGp6NR2GS4rPw!Fb'
    'bN^Y#5HB1j4eDeW|mOh=q-k?O!+zxx7-'
    '=*&i%oS4!qwR=I+l&uiDpN=Nuvk0lDEAW#nWx|l)iNmWG#1fC>XTVIT$f#V2#C^|5DTBQhkn|}IbeBo_a3%@Vz&jbaM#b6X<V~9^'
    '*3aMNWgH9N503c3XSMH(sK(WjtJfyk~QWX+GpDeEgWW^U7CqKZ)kpLfoVNvrjUg>4qnvMw?Vvy9Gec;1QJ~jal10h7?VTH%d(quz'
    'B)bl7T6eWlv{FH06#CtY=sKLIH7j&+YBUx`+0r!+eQ{I&J=23z#_vc|MEawL%aQIWAawe>Z?FP|ExLK0E+p|{k*?OA`Csw^FB}KI'
    'o$*99I0Gkl0m)9P`|Dt2CN>`qyc-LXmu5Cd2o!SUN>OzDV@*3+rW<hAgD?;=UTjm9@iL;1$p|E*c%;VQQvPe~$5So8wMuO;r<t77'
    'b(uHBcTY4h;_KMiPAfF!BvjzMoFfYFtu8;UUr8Cm{J0zsZS_lW3V8=F+o8BZ#GAtykeAa2yG$0`v`_X2mq7SrK?pnM$-'
    'RiSbCt9_TfNotPdP3FeoipHvk8_bHdfr%D&Wevel&c4bQfUJ?0lZZJea6gnmC8$PxKJf3Ae^ck41GgWvUhmE`|wJq5ML9PJTI2zv'
    'S<Ux={f<ZuN`z*9Y!6J==9)O*NHS4t=L!Y^j|$z3H;H6;q`>i5F{w6TC0y}FvWZZj=QZs3=;i9Cvp+{##mZrL)dzWaF+^NS(G_5&'
    'MAUy6$1No&H$~^%)k>g(OT6xLX0rx0XLmi9#>)&UkW)E(rDTdd&?g)nlWZ@qii0%$X<Fb;<l%9VOwtElmJ31UZ`$(expkjBMXp>0'
    'QjHVtDOg)>d2b<kkFP`M2F!}7IV%(^P~Eci!N#n=^~cPB6w>VMkO`8)wWW(FCkjJsZ1918i#>u&E+DXDk5QDk|15`lPCIsv21ZHk'
    '*GjjT31#kf+Jl;ht4Y#dXG%!p@nplSvk9p0!~W{)^u4GLCASyvE>_})GHlQ()81{@+s0-'
    'MhCr8^4CoHq^6m+)=dX}a;Ab}!w1k1s*}~2CW22S&8bZACoHjyxkrb|WMrCuS?_Mlv@+iw;eW+a`r&=xGYkNpr9Wn?rZ`d=RKngZ'
    'BH4Va#yP?UA!<ncQXYwrsE{XnsF)=2`8R#H1Qk}`FRP?>0I}AL6vPy1NXgK3On9oa=1E6{Yz?W+(!3rU;LVPYZ7%3bM|YcibhnC|'
    '(e#ij$Od&;dP$Si8YM>eB2lpL&1M~2@2{)?2urA#?_PgDNcJ~x0c!ost5LB&EcUj0!-J-'
    '4w%1DXiT_n?VoKB5<WDX`p;?BC4kXEa$n<w1x23w@J6Q=q($QzkqQ>d70(7*s089)Fz_gxAa$9~WN2qdGH~=UegFkS@W`&-'
    'a=ki!61F$_lU|R0|<w0m+fYwyO?X%FMGmoSxgBwm4Mx#>^NJD5{jBx=o0g8eq^UEQ*9yXJe&x%zj3E@E)43Z@Gbc5xCpT-_fTP@3'
    'V<E6B0c~DJ~T*w-'
    'ARBKzub^L8&Y|}t%?(#3zfzmCz?^9S$YjkJPM4!Y{oH@xDbzQ^ly$bn0X(Uz+KyBKwz@64n@ZDRcAET06Km9OIdM;N>O`?%hE0#_'
    'jcbF-<(xPcW1an>EUD4E71pp`3bB#Aef-uWuGlC$!A*p!vm{Go_<+bsOB!(fCr@Fc#%W*bQ{Hkh$Xg8J0TJ1giwIbCy^rc8FEVl-'
    '4B*sep>in(iZJl!dIR70Tp(qZF0JX?ufbdxWPC~f>vEZU-'
    '7+^}3tx^AoIgaI$9b(!e&%uM^ZO)A#pjuTaPUH|;Qz$1IW^1NJ>SP;`*%nX8sJRISVfR^adROL!WRS%rlRK&hNuje-'
    '<ba4{LziVIgP`qsq7JpjCrBr;SpgqQitoDToU*2*<x3s2Q=Xnu@RU<6q|ls!LNd+P@Ekfzr}&AuzG<FxE{wJ`9oRyWK$k9k#EvBJ'
    'h_GT8TBejpVqrL+3@=emNUe<<m(3>Hg)2xiznaQCu(4&7g*|3A2|x&JV@NZUnu8@3qO#jXPxQ&kDAH_}P-'
    'K@jK`xjF4{wjDaK*%ErGelZDTrI3F|f?HZ4f+bAX_%Kwkj6ft=2RbdHV9YwV=i={bt)LB))y@lC@lig>OqMiPKp4sA=F~=+ljG=}'
    'd3>oy~GdQHx4gZz(XKtuSrQ?Gce1h_*>2KsDP(mKs1ecBl#=v%?xXry&{7@XdJ?IQ)6}tSwbcra(np%GFWzIrM=L-'
    'XxHemrp(7s29W@$Gc(rGjT&?#!G6!Y-'
    'J7b>xs&j>AcFvoC}!vC#tl!ZxH(5B&z!`<&QBYk5?Z4y|vPh^r@~PXjSx!fQ7jh2nhzU13ZB)7Ty!6rwKD<e?~Y77F1?SLj|I-'
    '3;<E^DQ1(}*t#5WK~N6zjHYX7ewAyQr7TM<TL(rNb{O6y&qX>g)@(SL`n5)4T#C>(zLHtSHgd~|2p>TrRuYN2r`GpT=^uzxmo5q1'
    'c1oA8Ut+qPe}XXdGAVH<u8i}{y|bg{?VD{VJVCh7oK_2@!kJq|6R3UC-Y#v<?KefKu_s7#2i-'
    'rk_^6WC*4|6Br%x0YcS>WE9z2fRaw3HZJaVO84UZt~+_P(F!AvVEcQk22%ZO60AY-'
    '~~y~+yKO1{?#rouhcDf=1XKeW2CvZfW?bu?a_A*<{=sqZov3wq6~)((B{tJv`fB1wK|h2zh)DAF^uT8%zlXCkzk4`Os8F=w`s<|6'
    'FHn0><Y$xOXRlrr*`>QMPDX2xOj4M~~tX{#M8>2lQx`h^ojpb|V|!IRq3AvwZCns^=y(+li+3l`V2_O;yaQM;yC7ATz!o?hQmTzv'
    ')h?g^Oc<r35d`RQ_d>s;bD&SjR9*KIYgaSC`XrHNa6M3-'
    'tnlaV|{zoM!4F4f(+{c+<q*J52n=M*&SM@*q0%Bftay}3j<2vjM_E0l_jEL3%Db__~VDpDQ3b9|y4H?QjeX>B<&^U@=I8watPNAO'
    '^>!~yeI-F#)IX@DSwoUyzl!;AhH19<U80sDu^n4xWHvmy?SL7-'
    'VAN_lDD5xFa`Smq(nJp<WKZBljZD|9#<n;+DAdeW}#nl$N><5)7uZ9dT#7m=ELg5NQ#78tsE>uqY&kXuw3EX(^+w4c=|p*DA!AaT'
    ';A=D47EEoIN)MoTmu6aA#rl-zj6E<hA-Q<+Fv(JH@lTC?@wp(>T}2en<fJ(m!+U=5sza`Nz@wsEF_#&$8)<oLjwerP^*(%gw-'
    'D`W}%O5MRf`4e%8{9+Ie%x8<r1KHxV2DE@=x9n3+;&1xOY#DK;`3u~uSxyFdXD-'
    'w+ov}8Kxz(UB1k)u>2h(JeSiw5b3jIuPmL|wXY6OMz_7VQ%B2k^1eX}5%>ffbUNoC%8lW3ikRmD+pep&h!jR5UCO^JZc8GNe*IW_'
    '6$NoY3=nc<>nyoD1q5LXL1bZU8A$FNvWa9%OaBH!Sbr4_m|aqw!J==5zNc0Ce~j|aMPei!kuF8JE^OnFsrVoqO3k+uT<D<jA$&ca'
    'r}Ktqa~TY<p&Cr!<cybL>{xA$%K1K_!6?XU%jaz=pSe0}`3ebUt&?*%JL`vFwiWV@vZtM)at>=ReaD8c$GsfUDf(M25#3NN}mq|j'
    '@+0Up;<Z@XA?Bg+h8IMCBKDP2^C0LhTOZKFePS6!J&t1H}-y;-'
    'Kht1LQ&w&wc8V3*Hf;l(&tD#M48qs$d0>~B>tNBK=d#2rKuS5N}9X{Lcy$x%e9QfPqPr>?9r6T6EoxyWI*0w5@^G_SYOt0Q!bCB@'
    '6BP%r`N#zHZ#K42Xy3$vRnmObYJtNSk9inoak=Wz=nAQF1d$CuFA(2`ncLvSB?MV#-'
    'mQv@d|4wcnz?b@P%?O=+iv62P0bQb<tPU|0bod6^_E*$FpBn~1)Srjd^q{>N(qN~xe+OZQIkz9!%5m2EOU@R)piemYLNV*jZ8ajI'
    'YSF}RMzl-17wYI0|tsK<L*d5?bvD~jyFC=6_MigI!NWw07<x#=qM2RK<5@Ibdx41moJbSGS9blPN!?QDmnmcCBLhz%YffK-'
    '}8fQw7j1C7sH&KtuUeDo|I9?$TmHaw|BW(?-UZEb)WQpDZXH7!KwXB#^STu`Ac@H!J-'
    'DF@aqme{bgYe&MX=zbeR!05w)l%~n+s8h=FsGOHiX0Cy^hFN_+vQ0ZZ0p~h!B(!)TZb8jWEEV;t~#`wQe%Y)f9lp!$HE<4N`R}wV'
    '>O`#TI>J~R8ba&Zkx_McEAn@c^}$9TY7~cN5*ag)tpK%_1dxrf{GBe;wFs{NhJzq#4>1Exq){ifP?>^*c0ipC7HEEn%9SrOllL0('
    'I}ftOdJTlLHPsv_C}AjBOxqV-UA#o4RWcgMXc2uv9{$WFe*(h%hb@a)+nBg9zh}&l#)~?T|yhgmN*$j9D7*gq#wtW*3XDASu>kv0'
    'e~aTpJVtugjIV|HGnqJ1i)sO#71+Vvy-A~Bq1xLd$|p$e%#&Nzeh~SbVU?A;8``ht?&r-'
    'oT4~a5LXKQNn2h9U%&Qi^R~VCPwxN1v2Rv(-PY-45^A&~r~tq&YWm;xu2U{QDha}w{QxwP*nXk}a<epbj^&o{aEF;Fjtjw};-'
    'zY$Zf!Ah%Sq8L$LZRfgwwTonefjl!E*7LORPInFPhe6Cv-^Am-hejVrg8o9*&qL&*MsLqdvSYsVYwKRnsi4R2jl!7vu`e(Q$TJ{S'
    '#a_KW*Q2+xb*2k0GLU&h=}}YDHO_$EGgS$s$t9UZP=cjlP!QG|@iQOv2ZLKvdb}242(UV_V~EDDwtmNviimIzjc>rLTkv3#}Qotn'
    '!`aK*j5Lo!F<+I;tQ`yt+PI!WR(E=K3+w=!tV~qQX@A$z{}q%))BV?uKr`@hK8Ncr_ZGcHT*p5GgMsfUu=T2|*Eh;tXe{O#$g6ju'
    'ufra;XhD+)1hg9D5e*L)Jw#&D+~yy%$q=qz;FlywD#Mg}K5{U_YG_&6nXZG`%@PF|X6IW-{Uke}oel-'
    'm0^?3B?MlUaD1klP{eaMc$}TMK1N|NfbBjeA1dpy5N}|e<eYXTbFP-xHqIXmP<~PB^5*!`cSUy`O=#UtTaD+_#NWdMF;fo@+2r>X'
    'PGk*@NAbq)5u|Ebg&8e1Da<P5b^=8Y~^1QQ_L3s1)Zeo6^c#Ufrz4qJcPD!<~j_9l4cAg$SS#)*kOSKV(}VmUs4*GVH#*mF~ksz{'
    'ind8XC+i}=*oy=bPKYFQs(4&Qz&=$mD^DTZV(W4)x?G3D&KhB!DIC1-'
    'M@OCBo~(fbJJrTQp~ICN!v9}C?rPd<_xJ*?F+57QLNj)@(MhxKYRWXd`JyYsPJT2t7~su0W~JrB4Y5>rHwf=|6z&3nc)M;=UElyq'
    'a9#~&4rf*=E5;f(8z+R_>2$c8n1*<b5u~YXZVz;Ma76=r`1je%&ZDC)sp?XDD(@eGyw_)76b^O;nS$SCEe<#g{j7r+9D!+-'
    '+p_sS9v%d!x#BTfHXc2-(oETp1;UqT9$3<cf-'
    'V!+M$)zT2<R(e6!`0x4Lx=B;Cd!=&*JFbJ<GUTJuhYCso(7#ZYP5*7`+gQjsv9b(HEt%W9j-'
    'rlPmJ&y^a`?m)_3?!&@@I1p9I(mE4Tku<K_+pMcadi^n$53NiT6u_5P)=ITflvz7u2M1MWX{B#jHB0N2zR$H*r_G!hA5fj|s*PsG'
    'fYvCsleGASjk<2)CMB9wBbuY6z>!o)s}bt7ng#Y?rn&**)n%x-'
    '((_#<(dDGIoMICc0zNzTVyg#fzs7rzg%gp*bL^N3{HbP)ov_zpn{3Dqn~CD!Kr4pMj9DOYsID$DWI<<Q43y+&ke0L{zpS$cJ;nvE'
    'u=NU<0wl6lW~-'
    'w}(T*!HXy~8XG`)DHAS$aRWtutFQm*n<VbmPXKtT;Id^bsg`^qAfO5l`diO|lI(qsDwQck3_=6~YhK@2vBbq3KkJN}(rR+4*^gxC'
    'xtslCr`F6RL1%LACy8bR!;5bou5UtRw-'
    'CwvX>P}9+;10Z?z`&EQ)7`rt;Hw9ExIvW})RC6(DLvzeBIZVPA)5U_zvBCf_XmOo9D@-ZbuMvY-JljAI6S9XjnwYd9g25#kBV~N('
    'RxFv5CrqHq+{!+CgXV_=7B`8^<T|N@Y_%l+PKrH)q$LFfT!bYxU<P{f9Zp?cyiCJp(sg>NGB_Bs-'
    'QOW6h_z%QTU_5Js1kE0lE`9#!h=zn5S3UONIZ%wp}aIW$`aKHI$_S%AGpw9ZkOf!YY(50o2;1t11)i4MyP~tC-Tb7j@qk~(Op#S('
    '?dm^9z1ars?I?lxhzY)*JEYoAR(F^Itm%7g}0{GtE$3Eq`U?mRY=_L;+~7mZ4=S>C<%;l)nax|*d~0&v8h3Il+Pxso`T;NSZr72)'
    '|TOZlVgRXf?Oxop>4m|PoG`nLtA2Yfh}g|Aq&p$eWg7160Z##V@6wp25q{6Qrk6#n;dq~g3P$fmxEB9UbCz>`R$t&DqDqot8Yq&E'
    'Ikt+L|967OrpEh=dldija5W4sv8&lNzieY+Exq_mEN6HnoDoJw<RxtlE4^meMlF`(obT@iOp@J^W0dqWgI_-CE0R8Xei#NHig^@<'
    'S)K1QomJ^<dsHu()w_b(vzU?N7jV#xtu-'
    '>T}8zRkqyZZhEn!r^{v$RbI?}Ujy9B(q&3DEY8ipLR96p1bxJ^sa=wKSrX&=srZ2_{5!vg9@vyYb#lBVQT!<S=3se)E97uN%VXIu'
    'kO6__`>I0fc0Gf=)*l7W{jAdg<x+pE_+sA#VQcpR{ywE<I{28W2CbaIPhwxk2d55ybG;LW%Q7WuMpshy-JRJGq+l9_Hq*HBTidb0'
    'cDbNAo+e}ww0Z?+LU8LIR#5TOFwE>VjBmh&$jUo@>>9vDit-'
    '7LeHwKh@g*yv?x{pSD?)v4l4(Lx5Tc1QGtSl$rkrxKJR2du&Dx7u3<O&$cP?_GL(p0J0Y}6DZj#paVU&}W3))-'
    '^YgQ%X>HeALU3zFAtlTc(W6=XGasDz;!hslCEvxK;INh%wjES89o3#ou`O32wuNv9<=RGz0|2QVxela%HH+->VYFdr5j2g-'
    'p`Wpcz|sfm2kU^x;+3CaRceejTd%}dz<G&t%mTLXBQ#rFiGYFq(IrRCJfu#M|3r4gNDTltSGIkKs3HN&@w*=ewOfIrc_&c?2X*<5'
    'QP>R`H0ky@x|GwWPYCAr01JiP%dVV@`C)3E*M`XY`40=FU|y1lRR_F#k`vUT=JWqaF!(^7OY{UX~{!nQUp-'
    '_}(ZvNep50cTGDBJ7qYV;UX`$k-'
    'JsLRq8~6%X3^rKI5MLU(q`u60y#fMG82(F@*A<B}g<l#5MX6v$5@VVjjwH0h#pTdb(xQSR&y!AZ;GI?c6?oe5>hF)oTxDJ4EMy>k'
    'Mwd!<QOeT;=j*4k>+KbQ}mm9%H%vQC9MvBEm4dX*96m+BO)G_^xjq1t3(ooE9blOtLYDgq__0p~ckXL)?)Z(yyhrlJN{<kMKw@yv'
    'l<phq>%vRI>$wXpj)4AEp@LlXTcwXWnWQ;{7R)xwoF1TqdP+6vI+=n$cdi0r1=OS;ezqDZ#_Z|qS+dXQNo3{7nbrWNUSFf^J(0E4'
    'Q9rJ0=!FhRLENyPrR&OV!SV?RO}R1-C|0GopmQJ<sIb4Z+4h!1cpDMCe5B|}r^*ZUag;6-zVDS<pyO?Zi6oKxnLJNU7-'
    '072m*m>$QcXh%MSHylefSvJg7)0T|@F@vW>XsAxR0)!|2o3f`}u=EySSlG@t;BmTaf?w>WO|;-'
    '*TC#@zB7zhaaFjrYwze64TcpNWkV<Go;P{56SvM>C?kllLrp-sVA#;{RMM%X?t5Tc$@NK8cB_{&ajCt|Rh=bZAQOGrTGTd05h7t('
    'e>aBX2rccNorthpCSP92%%Wpl_QQoEuiQCL8?IgZbkg5_{$(&x!U&M-'
    'E3qmT8M53(<quegI1<6C&WeSr~H?Fh*pi0Mu9e!HO8+I;Bcp1Al%O&efQ=Y&+L`I`Fwr~CVHV66mkH`N9*sTax'
    )
)).decode("utf-8"))
_PRICE_FLOOR = 1
_MARKET_PARAMS = {
    "WHEAT": (25, 10000, 400, "sqrt", 0.8, "log", 0.2),
    "CARROT": (35, 10000, 450, "log", 0.2, "sqrt", 0.7),
    "TOMATO": (60, 10000, 200, "linear", 0.4, "sqrt", 0.6),
    "STRAWBERRY": (120, 10000, 100, "sqrt", 0.7, "linear", 1.6),
    "MELON": (250, 10000, 300, "log", 0.2, "sq", 3.6),
    "EGG": (50, 10000, 332, "linear", 0.4, "log", 0.2),
    "MILK": (160, 10000, 122, "sqrt", 0.6, "linear", 1.6),
    "WOOL": (200, 10000, 105, "log", 0.2, "sq", 3.2),
    "FERTILIZER": (100, 10000, 200, "linear", 0.4, "linear", 0.4),
}
_WEED_STATE = {0: {}, 1: {}}
_WEED_REPLAY_STEPS = 8


def _get(value, key, default=None):
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, "get", None)
    if callable(getter):
        return getter(key, default)
    return getattr(value, key, default)


def _copy_action(action):
    action = copy.deepcopy(action or {})
    return {
        "farmer": list(action.get("farmer") or ["PASS"]),
        "hands": [list(order or ["PASS"]) for order in (action.get("hands") or [])],
        "market": [list(order) for order in (action.get("market") or [])],
    }


def _seat(obs):
    return 1 if int(_get(obs, "player", 0) or 0) == 1 else 0


def _farm(obs, seat):
    farms = list(_get(obs, "farms", []) or [])
    return farms[seat] if seat < len(farms) else {}


def _align_hands(action, obs):
    action = _copy_action(action)
    expected = len(_get(_farm(obs, _seat(obs)), "hands", []) or [])
    hands = list(action.get("hands") or [])
    if len(hands) < expected:
        hands.extend([["PASS"] for _ in range(expected - len(hands))])
    action["hands"] = [list(order or ["PASS"]) for order in hands[:expected]]
    return action


def _tile_at(farm, position):
    try:
        x, y = int(position[0]), int(position[1])
        return (_get(farm, "tiles", []) or [])[y][x]
    except (IndexError, TypeError, ValueError):
        return "LOCKED"


def _trace_actor_action(actions, step, actor):
    trace = actions[min(max(int(step), 0), len(actions) - 1)] or {}
    if actor == "farmer":
        return list(trace.get("farmer") or ["PASS"])
    hands = trace.get("hands", []) or []
    return list(hands[actor] if actor < len(hands) else ["PASS"])


def _weed_repair_action(obs, action, actions, step):
    action = _align_hands(action, obs)
    seat = _seat(obs)
    game = _WEED_STATE[seat]
    if step == 0 or step < game.get("last_step", -1):
        game = {"last_step": step, "active": {}}
        _WEED_STATE[seat] = game
    game["last_step"] = step
    farm = _farm(obs, seat)
    positions = [_get(farm, "farmer"), *list(_get(farm, "hands", []) or [])]
    unit_actions = [action.get("farmer", ["PASS"]), *list(action.get("hands") or [])]
    active = game["active"]

    for actor, transaction in list(active.items()):
        index = 0 if actor == "farmer" else int(actor) + 1
        if index >= len(unit_actions):
            active.pop(actor, None)
            continue
        age = step - transaction["start"]
        if age == 1:
            unit_actions[index] = list(transaction["intended"])
        elif 2 <= age <= 1 + _WEED_REPLAY_STEPS:
            unit_actions[index] = _trace_actor_action(actions, step - 1, actor)
        else:
            active.pop(actor, None)

    for index, (position, intended) in enumerate(zip(positions, unit_actions)):
        actor = "farmer" if index == 0 else index - 1
        if actor in active or not isinstance(intended, list) or not intended:
            continue
        if intended[0] not in ("BUILD_PASTURE", "PLANT"):
            continue
        tile = _tile_at(farm, position)
        if not isinstance(tile, dict) or tile.get("kind") != "WEED":
            continue
        active[actor] = {"start": step, "intended": list(intended)}
        unit_actions[index] = ["DIG"]

    action["farmer"] = unit_actions[0] if unit_actions else ["PASS"]
    action["hands"] = unit_actions[1:]
    return _align_hands(action, obs)


def _shape(name, value):
    value = max(0.0, float(value))
    if name == "linear":
        return value
    if name == "sq":
        return value * value
    if name == "sqrt":
        return math.sqrt(value)
    if name == "log":
        return math.log1p(value)
    if name == "log10":
        return math.log10(1.0 + value)
    raise ValueError(name)


def _market_price(item, inventory):
    base, equilibrium, scale, below_func, below_target, above_func, above_target = (
        _MARKET_PARAMS[item]
    )
    if inventory < equilibrium:
        amplitude = below_target * base / _shape(below_func, scale)
        price = base + amplitude * _shape(below_func, equilibrium - inventory)
    else:
        amplitude = above_target * base / _shape(above_func, scale)
        price = base - amplitude * _shape(above_func, inventory - equilibrium)
    return max(_PRICE_FLOOR, int(round(price)))


def _is_sell(order):
    return (
        isinstance(order, (list, tuple))
        and len(order) >= 3
        and order[0] == "SELL"
        and order[1] in _MARKET_PARAMS
    )


def _impact_score(obs, order):
    if not _is_sell(order):
        return float("-inf")
    item = str(order[1])
    try:
        quantity = max(0, int(order[2]))
    except (TypeError, ValueError):
        return 0.0
    market = _get(obs, "market", {}) or {}
    inventory = _get(market, "inventory", {}) or {}
    prices = _get(market, "prices", {}) or {}
    current_inventory = int(_get(inventory, item, 10000) or 0)
    current_quote = float(
        _get(prices, item, _market_price(item, current_inventory)) or 0
    )
    later_quote = float(_market_price(item, current_inventory + quantity))
    return float(quantity) * max(0.0, current_quote - later_quote)


def _impact_slots(obs, action):
    action = _copy_action(action)
    market = list(action.get("market") or [])
    rows = [
        (_impact_score(obs, order), -index, list(order))
        for index, order in enumerate(market)
        if _is_sell(order)
    ]
    if len(rows) < 2:
        return action
    rows.sort(reverse=True)
    ranked = iter(row[2] for row in rows)
    action["market"] = [next(ranked) if _is_sell(order) else order for order in market]
    return action


def agent(obs):
    try:
        step = min(max(0, int(_get(obs, "step", 0) or 0)), len(_ACTIONS) - 1)
        action = _weed_repair_action(obs, _copy_action(_ACTIONS[step]), _ACTIONS, step)
        return _align_hands(_impact_slots(obs, action), obs)
    except Exception:
        farm = _farm(obs, _seat(obs))
        return {
            "farmer": ["PASS"],
            "hands": [["PASS"] for _ in (_get(farm, "hands", []) or [])],
            "market": [],
        }


def _kaggle_submission_entrypoint(obs):
    return agent(obs)

'''

PREMIUM_SOURCE = r'''SASH_ACTIONS_B85 = 'c-rk<O>Z1oa{Mnm^B^`!t;RR5)awyeGZG}t5^I4NEZ{W^80*8>H^cwk(vaQNRT&u(neVkmyYOjxTI{O#{W2pXBR~Dm#lQXKm%sh>my3V;bn*M2UcY(u^SiqbAAfqkzj(O3`1im3=fD2f=YRS9@o#_m<v;%V-=9B!`LjR&eD~w)AMV~<Twc6<dw+3x{cySY`os78{kx0HtHVEh*zaF`{`!ago3}q+T>ftJ_5JtzyN{p$`q|<8ckkc6`swAzlYjd4C*QyRwOxk~5C404+Wa5izW@0BX|q3F-0wep{PhP<|F-Uk`;^01pDy0Le)-3r-W|GqarNt$k1y#xdOPHwU-9Pd<^JgnCoPBX**yRAPk&s-Ea}2=NRA(3r^q|*?>_F|t3JeU=1j!#Df_!a+c#Z~<9F=c(@GNkJ3R1lrLNu%-UXh0xs1`Liw|#q+OC|Zolzg=<*CbLij@fld;6Zp5rr$|4_`JX9%MG1^#MKn)2EA<cZc<I>};6Nr~iK($A@?{#k1p~GPui>p<#X;lhT03eYRdLaTJcfcpQ#S7s;qET|G`PI1}zZd?+u_b+fbGx&0=*$)3hI)XS<#cIL68%U`OVlA&DI$}({IUg?Z!eB3f=r&~l0YGLe_>HGC5Vzef#$m4tBhm!-eGkW=h8)NqLTkrXm1y*_P=EHCB*zL}`;g;gY#M5Tn(*ifGIBhf}N5RXtZ{F-*e*F0l`}ZGTzj^&HUuKIuIJWAzO<Jv54$IMY91nzeTWo%dUbX5G`uAr4j-w-61@<koc{^r%VF8(jm$b!+9k4u4KZUjCXaqlf_1m^)`@m*v)*nVM%ai7HS`|C@bRX5)bz&wI+H?37d{E(D1kX9HoNwzfi&v+f^7NNu52@T_n=9)}pZ9OR039c1<>k%TRfdp~s}vErc{tnE=bCh+d&76p@m5O*O1BSPOZbW_I9h$P#W~f~uz+Xwoy4a*ntJ5H2`-vB+XJtA2_pb&>_{KF_s-ay))fOT(BE)$kY7CFkrtdO+M2O@3c`ASU$^g+d-EUi^8NqI)V=8ZTb14&{c+hk-ZMrBVZC!LkCPi?gR~jmt><~2$31O6E_DpMo@RVZ@T*2!ctpW^IKfwF;gEk!{LuLx>GVr>_BA<p)lGif=H~nsp7!ef+jsWU;GG<U0JCrX>#s%v$f9>F&{Q)pP~^21yas!IL5FQ{w`>uhZ5(2O$@>w&J8UpzrrO<#jWESrh$r)0vw=lx^)2=N-TOa<Yi0Q|pA~j148&){syJ=S@pB4_efaqP?*8}t_wWB4sYR_|zjLpmx9lF|>3P~S6n0)?)yA20_{GT}NCWe<5rNoX4)fS}MoivDKK15Ujd5<$SW&33SYylKV)a~xkwv;ZejD*Atk>-9W_*>Dy|KW9ON)k>BXsUqSf>Fi(LDhkK8!l$<G9+&rb%h-6%*PUvbAZSftlQ}Nw?il2IAIIJcLS~xtI|+1q6{O-SKON>{ZT?-83O3;R>^`ZY_3z97SUs3C%=UrxgUoW3#cIx|Vq|KxZN)26L$a#6&J`IO@8ad@wcym~0^S;Ui-4E}{;>9Fta3fB`45$+OP+EG6F-cxzxLIw&Ba_$B)W-3}YR?<{;B4CrxNAz+Nqs@zvnyex9rkIi}fW>P1QC{7+uyFC1r1at!20v$}i-kb#HfhRAQ;*<x+*%*P);$s_(%QHPToLtWvSlrWZuM-37YLpSvM(1|FJ?ONX9K1Ud!bHbt==_U=1t3mMcYtS%!ZqzF-c``d&Y%=f)8lqV;YfM?KArR;s#zGX!ybS8_Va(9yIVZ+&VeoA(}UyVmcB6Cc95<FK$lK76+mn_<k}1nZbRV{XjapMm;e))6=$8qdTQ~O%s8w4Kr_lZixUOrr;%EMS~x%kdd_o7KKmK3E&`&m04-MN@K#?}$(3H^?gCi-21hS$aO=zYu`C<(e8e0?PHi!RV1L`x*!f+L#Mx-1CBVQ5l$LRAr$Bc1B=*G9YldlAU?M`k_6GUZJy)|I!Jp%lpr}bds!8VFfi7S?fSDNcEqky(Vz6a1LH>Xlm^yeZG<fIm#b9Jd1z<XMMBZ{0&YGGAX0P}mN?<aATY#q9DNa=@yoGslhSB2nge9VbOy(r@*s+x2u?vtjI%M({@qhzQsnOsD0Mj@hO#5RBQ82!#GmM0tw?SP^_zoJel88bAlZi%8AlZ;Os}8}^a4WQtJXZE)m62B=y<8W9vQpkd*@4TlkU^@l9{?`FqqP#j6P6~>VyeYgVA{;-x6KIiu1c5EWS-?{i~Tt6T)3>B*8UH#-~90la8`j`q8+<^Rc=^jaii^y-(}P}uz#~YkSp)lV8vGev6gil-C7QthPpfpK_^4+z0}@TW)|V6o&Ic;KR{^<aaBp;r{+9ZG4(E`FhoaJtkrd-6fUKqoRu7ZcbtdqxYvx})hZ18fw~rP6grr7<^o2u84Va{^4cSxzeXTDDrU<G>5S`5QDTMA1A`47r|KlT8M7i$`|JA(ATjK=wetJAwA!TmOzV!R`K7EMHA=t%*~G+FFNADzKzQopa7DkwQU0@*hBZqx3+&h?d0|ETu>$ANsLRp_z!(a48q3EvL#VlsCumf>1fyR$kRxtWTfs4Y+xH(!Szmx6bJ(G$zgx#|xHGw9UyCThmK@`^)_S|!sM-_Ak|6Fcu#dKG<mp$NtDs331#j@p)zgqzW|0JQ`+7iIhDpv~^KG%?dgRCJ3i<IqzgObLZD*`os01)!scY*bYK<|i*&+^Js)+MQgOa*zuU`MN<#EC-kUK$X)TWRi#8@XX#^T%K*4kGVV9v-nK3A?u>T`AU4FS(<(;Axmm2B7&7|qU7613XP8Zi_{U0)fzJjEOF*vUsU#ti@sadi+_qM%t~2FO_akB8f9^ujn66cM2NXjYI{It3sm1~wnim9M2z#j!a~_<C^ygWQe1gHE(F5{b?-YFO%YG+Y}(9b?U+V9&;JZG{U$yGU@3k5TleCAs#-j+j+Mok5TmtaR2$bL_%q*;$&<RyzM#1CV|u!8D(-YY*057DF#9KlrNS1r`I1r<Xa&z8r{eL>oqB1)Na?MVK>&X%nLw4c)eDgML*Sn`p~KuH)2%wFrX(H`D_IJ1v6QCcsos!xd+9fZPYrNtz96j~JS{vc?ZO8`X>BEYHf(rTIISyaYpe^(ju_Y=GUTv|JI#OD0Mv(Iie2^hk3UPme-OM#|%(myJv%A<Dkc0v@^e&lSn3YbDZH8L|_VAXb{wl%+vb0vXVXQ9dHw=L3ftL6vg?tNi5YY|42BE0X{)_vo4jtfHEAMN_xwnTDJs%2YbpB`ZEnp4gl3r(KDff&<UYu*}2#UE4a<RZ3RY{1%|8Tc%&w961(5m{NAiHi1y_+=UuTo0H{u0v<bf{E$e%tyk&Q&Q3+GI;QK$=8i1#=My5EgURh8$*UPt^dCm&=tVL|FTfE~*(kDqP=bzEgxnQPEK-9B*h(>^lr&(f+t*4565JT|@9GazEW+sEt<gb7FM{+zWs8w4b$1>f!+2J<Sa29wB3&Vfp_tHqgpon#X~47w|5YNBUivy9Ci{u1kl`pD)hL|cS;Qd3!lX4732h@`)Fp^|!)$wNGQywnAv!(=8&MFw20mO8$Vu6^{fH-$$q<7|18?kz=#L*m3exbn5tB5MbCu|&l_japd>RIi@QvO-9FXz7@3@<#4?ngxdFHs<_&kG?Y7knp4M=o(A!S860f?U!5$#6r3z1$XHJ<COR^^ny>B|RzHD-(kO{TGsz^!H+`mjv`GfjqbQ_a;lr@`EEb1il;4QwMoF*@^HX2MBk^+*h$fwB+=#QmcPqQn!t;>S;C*e$^muBXSp2hjyI$1`clRW0Y!h4LNHcrMA~gd?U9^;d2@feCwcoq<6gKM)ENi2U0kaMgMeCs{l?&q(N_EVCoXxTk+Tzx;J<PNYuzGH-x(-B9MB3Mv+dxlvJy2tJ8)O@SN9us$>6W)xFRE2w3qN}?_9>*h4Hq-d~M`O__~r4JvnggMSlCz)HtiMG-k(rh^u>>MnSMs6K&vV&=JHPo8+Hg`I7fuKAoyE1U`a9MfDk;cn|+Nv1NM8Pp87vnIV+ogGSTAB<<&OD`+i1I`{GFjZ7R?cYw%phBurhE?9k|H(oPAJGY8Nrk3qL8N!)=Lsjbc4)P6Y8X@^>9wMGsqiv84+-*N-Pn_tK_tSoMJYTKU^9&s?f`9q=rsxKl6aeUDO3;YmWQ%nNyr+5dktsbaHyJPX<^tdBC!L9=&*jd{UqSu>50|AKvhUvIOU0PC86#alG={+hij{$b8J`jOMxW{ir|VU`l>En;Zj|`b~+^8WW1wH=70!#AFmf)()lq#WeXD`e2kzl>w#FDW6QRS3E+RJws>|%V+UJli<eInZ(-CDX1Gnqbq9w4J3@fP)vE+Pl`ntp3jm{?jm15_g)oKWP=qUvEU-|5}4aEEq3PdpyCb++{;{F<u?+iwh8Gl!UHo{6XCSVOvcNBy^z8WWjA`X3=HQ6$f&*J-j%!(HgePh;U1lT8L+{kq4^R$zUkz~U@CeA2>a5O%u66=?PE9-SnUg(lTY2(X=dD$AY;3J##U>V%nCziRyg$Bz7>y`mUc^g4YW2&Uabl*vusb4jMh-qh(0WgeSz0vuzi+_6$|Mf8xwO$GPGO9s;Rb{;Pzg(%hx~^;!Hf;W!d!^pjOsDurnChy0$n%&B>>s*Wu~N{Fd^cic?JmOUumg;u^s^<qbw_Ou^#CB`rL!kVi{xe?Op2^Y-=*;kPG7Oag6;N)CXTVrs3Qn&<Na6a_tzW9SIQ#4U>2iwGCoOEqN&hV(rk$O<xaS&j@nJgnS|G_Tz#TqO<{QGzL+rSKNA<=cq;pPi0(M0^w)9uAo!anJ`;!Fj%vr!miR4=NmiW|D*_E`kR=2XSgD^sJhZ4-dIvPEEp#0~eh&hR_C#cnJ|p+WOX0>{SaNkx-CW%$z1NCpmCfJiff1+V+r<;$92MON+1dnkKL(E6yLq%e!@_x)Jv20K;ONuIsJ-<3-n3w)ZnC-ErXEbV|53g6o~~u9kD}>u`0G-L=t8g8z1cJX4YNOQV<K{=!Lw2oC@#!MbQ<^IsQ=gxZ;#1q;B=f0$4VnTP>{v_s$r>4=6h?FlJ`JJUJr(GvS}7pu)ip&4nM;oy0!WL>JZxju)kpVArGR6<U99<87e!`{pl#;e!15|hLX5b)BftTQ4==oUWq=m0I+KS?fOi=k6=i0WE!ioE$rpOLW*l*r&J+YXHax6K*tBhm&~Iq7CXy_*hT!Yn7epc1bk5O<Jo#a8Otwuy4=Cf1!8(InD~1}nKXjt&?NvYX!7mam*mxg~pu;vAuXaT>~=8aekCfX8(|8)y6+9<3=urd&>13+pSmv8_-#mn*y}smHNI&l*Sk^h_y;!B}bDAdIhIPO29pLLmcQB{+5Az@)74f$B*zog28;Jx)A2K2kNF8ml%7ICQni++5MoB?{0Vq?%35+;LK^qHh-><E}yOx@gxrdc`UTv47*XGn7#E<Hk(a$Edxuue)v}D9i?i1<f!|!pOx}933y6*ZT^MT*tj+VNtWZZZe5^_2Tgib9~$3!~tLbxIc=t!h)c*;@EQpbSvRT&t>N2hw|dR6_8BiL`Q!Ojh2`FT=wxzBk@}M7)dz<;Ef%kkCWusEb6zx&{wO$=35WDj<F1}n~?2M2iiH%!F!C9YT!kh2~1jsmrOLMH5N86V6f8$MKv=R8Qt~w40E|@kQ|*0`*Y`VpZewNJ7FEzK9jc&7%ropHE>Hu3GUlRDwVbrETcjtBm8$&@QmNplN=#R@$%r7l2V#AKUJ+ZM??<zz@}3*U}j5q7WTqV6m*LXO`^+l<?v!Jfiant92=s$5|L0!64qVG3-pXeZ;h}h98($(C8X`O%~(l6q{5~Oj<O}y>G+OPAa)hu5_ACPQJEMEp$KlCRjUF!?HYytX=vu+Lejg({SCL)GL(?1UMuUs7|GRvkGfS!NbTOyr;c|N^(dJ<U7;v9L95n_c>RBJb*O6e_ynDi;Dzyi#$us*3q>V^0bYeNBY?w1@i31h0_t!mk0l~>o1)87*pRyY$&uklIuREZ%h^RAAFYoALc-zojnCMm-e)@qU=P%{Pj)eBq>Vj!fC`%g=2aIt=y<U>-K;9LlX~PuH99(la`m7R1KoiomDSJIv~<UC-2f1p9cY-X);^7~|6;aOz~2}7L9-wpZKtc}{W^{Xjz3ESW0?R2!_^WZVDCbE(J8xLDOR_erd$9$(Imm!cJHdnbYH8oAWQQtW^4=4y-(PWncJo+ILoi6%FeT4F)DSfsjQxEx67O-y7r{fA(f(z9v^~YYKE>D04X98Gw4hppa!vADoIALo--H;8uT>tnDJBgRvo2r?iM^R8JHpK3}zaNb4VB<b}Q45xDr3}Q$Yguq&$+uzp@T;rgW8izT>qhDKARKqYYtz61R$Ef@!Tzn&H$c?Lid)QVNclrBNb_b)^MO<w)KZwm$*fLeYKCN2blD8Rx96j2bdVIHJkqloLxkL$1+*8zV9(%=~yttMHQHOeAQhfp3BoPVE;G+Z>Q0h;U}LX#{7T+c2gp5z|`0CFLk3td$2$|GdO&`5Fk!)vi)={Tq;E99C+)F67nvWdJEPCC}}vNnq=^d-R?dG{!U~^#l5uy2tJ1i=j)Dpa%S8RqFvV;=5;*7Sffl@YJ0%Z+nu=f#s*uBoU@)*pk4Aa91A&G_x*vty>RR1NLDJ4;=YRF!=f$Vuigr`jobdTL{>b3ihms2FnE3j&4o2U3jaGzc+zA|J^7H<J7lPf^b>x$yikx=vpYppT#jc`ApQmrtyA1$rJa9NuEtnX3Yq1aIF3O%N+Er4YM~Yu3T)Zs0Gz{V3}X_0;PjX(mOWrkgA1P$+@EyQ#~J+GP+v1|7^f$VhEG;=YQWQrU#m|uhbZDkc3i$A_#Nd20^#g4v0I|38`o(!<<r_!4U8G_Q)7lyRVSWV*L(*x@{~VmwI{p#ci^bZkySo3e@lt?NR*DOgSP14ah;@K?e$fbc~O9d*YiW`>gB>=NW$V>?Y6kVqvNx8ex}RQ`Vz2Dm!Qn&#;I>hb;6@sM<s2Ogy4H8&RZoK|{E+Wa3Dw3S+52Kd7b1KHhZ6bK*K;M5NCK>UNd(PT8K+=MX6#XA0kwy%_f!AxUh3;nLwix$T0UG;#s-ccpMMsVY7<_$kLj`@fz*gk4qFEV*6ab3$mA3f($oYLk<%036WtG?CILv35o58B)owsHb3F8&T0ALJ=K|gZku@Y0+sC1EN9bmMOjhNyEtTBcc?UE6TY6x5l}nQdAfdB>oY7NK?eje;>%V029V49eSLbxK_dZvL=A+3)Wo}{TU$zu)G>n86$RXUI+5B0_R{j<Psgv&#tVQ2gc8wZ!-P|11To!>>HYBa##wLDEG-?Q+;cwfzZ+BrIX6#D40{?$$*&fONA4{>dD_b#A#-cS94^FnGcT%qZMICA(0?VbXzlIlp^9}d9o59!iww3nWp6@I-~_b=ka;@T<ye932HDgt|~wttLumkI{|gDT>2S#PD%nL^BEE;Lvapls-{}<0F7Sm9tJdvX)YJzhUA0Alpi<^t%hX6bAeSOXbX4%g@j(t19%xeoP{!vE6^+b7cO_966Gt%YL@pvF&AAak%Xp2+8?Zhv}8RdJRE~aQCk|6NT&f9V00<A(I{$_@?Hvn3@z7Q)zoS6J6si8HLlE2?|{OwoTIoi{UXOaE$G;4M>AW%nfy(R1`#a)Mv-VO2Hv;`eNlM}cC_;mO)#<+sg#WoKtTX$16DaD!_0K7GC@_95Y*~A8}~Q{=@P=&5W#;YG(FMiiOnYpCqUT;sTG+_4@U}jS7hXT4Ru)wT{Trw&d1W>_zzFgS#gnGgbj8}G08#{xIo%r`XkMpCLf(*ektmMP&uHsi7)e7R*H0a11v<$I1)8?HTk;I(L(<?LBgL0QzJ!J<!#5qy^2FdQUkCtXep#N|AT?M&HA~P)PbB#k!9ZQ9o$SH0*_~i=&nTL70pUj8QrTc#Hayo9VC?l)kk^>`Lf`wU{6Sr0{5l{imh*OMi|a4X#r$L3<8Y`L=2QJMtg=5cpJclQ>nL~YwO<eMiLd&hr{rP8<v}(w+@_auVOT`moPq23jO)1=$=e(226o`YR+Kd1z!2m6dQ~uM$>FLRC=@$a#Ua|g%ivUR@h}i!)Gt0kpz@P^Tj&Ia>Cy6S})6a$vil&JyoKp;ggmppI+K!%vs4Jt5y&Lmx=@@v$#6Q)lx~l*=Vs;w+2&=9)pa0n4N|MCQBkSWbaGh$*)<p)3wyhvx-UUuuKaJN@V1_(>c7IQAx^4+{7!5Z9Plpc~H=2cP6^q_1Q5)Bt2fQG4;(W{bc;Mgv@4_*JfL-C70Yv?V}rhOla7qq@{^+a%g>?P_GL{4%iS;h#^X{{!=A|k*%x|V%M%RNAZmj5Q6h0CXebB_FZ*_gV$5#=4OjkQci36mp1$p=MOVo^4zgb<Ug{l-~v&5g7PS`X)ejD30%}yVpoN*XujaI>OL^`K#>uWB4`$6F2X$7LF<?3)*{|&T^%h-nA0reZ)76Hb4M}V=N$nWG2(3Oebw7$5>VniQm%YLyE7rUvG$r*krLgCgbRX|!DaCnxqe~OM?5Yz+~N)zlEF7g;+TdaJL*XV(x{hFjHHP@A$TwYH{}_?3mEp&*TD1Q{7vNVUXWl-xR~OwTabDRX-vdCUFCP8-kz&>&l_t-3iOa2kKJZ47vfQ8@(O8NSe>i;{UCTK1-Cs-_9_bJX<io=%{L$cwfoK(yRXA8J6~>O4#ESCsBm3Z36cyUp!tlEkdcB)wLKmQ9ya@}6hdxR+rF-_k<|St{g#A#;mboe#KhHbHp_q4L`^C_l5}JDRnh4gYCTM}+zR*iIG9-ebU&!e1&z9`J5??x1^XljBSZoD*j*5Y0h;chyzpMdM6Y(O40MS<S{jwUrgj5-y(*PZudO4oEl>YBfu|$Z2iE4KUb6g4v2kUVs#8j-i6L+{A?I^pk^0#E$6dtj72ls0s&Vn05ox9;&fGekt0aXkUJLvPaNJlja#Rofc;5JmxAgU#0mPL24ug5$Or|0lw~DZ{IvGMbj@n}`65*Z|tRpcIpL(1oW||LuV~jf_Sl5u{M2+F$U9SpOAYn<-S`D3=4}DNrlTULMX`yjLw7XuUj<lJ}T(dT-62}BN=?v+(=Cy}*jAi3}dxf)JR2Vi>^{u}OI+8#k88PbSp&(V%Qi<|V6JJEwuCK+CeONBn{U8vstLO>|Wevw~*!RVEubXL?ivB}O4Q{)6){wJp)HB`%lROy8u>>pQa6Px{v%AKOjipe3H^K+H&QDH9%CD=EYsCwX@kW_-01Fh7S;q*=E3OE5NueTN);>PAL@_E1qN|T$33d#CKU|@RtEI%*qQ@iGk*dWGfT?F=pi<AOdjC#Z{Lr>e!U+J<B*hrpBTra#Jh<&*RLmze)Mrw8A7-SPRnBRG4g;tv+LP|laPN_ZsN}tNeLY>3cqq^eKrgKa;X)ljoEr_>GnFtJaxa5z`4XzT_|Z+EfG{blfUBuP`dqdCv}d_c_>v{OWs52zxf^W78175=#`G=p0IbGR->kR2UY$<I?`(SCmF9=4>U&X+ONieSDX{KxD`!D{__C{(Tl)6w=}&f|&)GzhCgwVcDsGpd=qp})-?c<Kgqo*U?4iyWFb-zm{*@4rcW>@~T;7PksfzcsY*dq78e69_6NE)INQN;|z%sF>&=c3J_S1E!_({n-v*m87MY*UQgwD?Zv@93ma6y89imjqLtri)`Q@aS~Ol(TL(1l7#1fG+<V}jw(hFM=~MU`$rS|(9>(3S2ko2Cck`d1L7h|ndC;p$^|B%3qaoS3HvmTJDm0NRx9{O#JhEFTDPo+9kZom!O@z-Qa!SDA@3)Q)y|%U7i)Qy)M2=B(sp(@?molu~t!8K`y<4+%&k%m1vLbYq;IKmlc6s`xIjnqsXoZ9$L*r}R$y?5DXmY??dmUaIrdRuowdB14>u$4j$<FwVr20o%pBTcr*%B=&R2ElYnniNv=9#|$8dh(5C%gjvq+M$__j_#v!|@ii*U#n(x3#pDMRS;P?sYBiPPq~0NJWPP!Y2-t3MgCf^Rb~###Tqg}BZ^cy!KrrKq^0Lf0m5nVPYFbe0TkOze_%5DA=g~tyM+H9#^ZDXP{b%}`M&HroG(Ta&>R$KX{PKNv=|4E6JulBB^xIX73}V_eh5&^)J+`@-%-3nMJnNhF)ytfuOuD*5Wn-+nsF*@jlQJc+dety*QOC;!Mu;kc?j4!s>j&S(pwh~0QF|+6j*ndtU#etT`q`kfGHXvgl!BNyF&hC;XMo@}vKJ|)C?W$|YDaTlk}=B?bPL&HLR_U=e+{8Gd0j(li_%B+CCd8>$fA&(1*5t}OA;YtB^WN2WRl#dwnefEU4?I{+8kZ1`#H2hTCtFErwwGaQjbFEaCz;__mQgZ&`VE^0o8ja0PkW1luzm_55UxqEbH*z)U6Wgd@Qx9UvjpSqU{yfi^v6+tz4(wm00Xdj3}Zn1U0J`3XF}HNCcH60&ofvA_$?FIM&CX9eJxoeQ2NIwKJo`-+=nh)w61$xn;5Vd0aHYLP=5_C3&2NHT@}aR*P9|zWkA*cY&NYfEV~rsazZMM#cxL>g7GDhQnn`%E5S6{7rQOFbygt3fmGFnX&B^)$kJ`Xae1A9UiZWiR$jzOV!!QOde;^D%mt%EJi8KmLpQL*obUJ5P5Cse$!gF(1CML4toK390E=lC}5gS1crq&)H2qdp5l^QA;iu$wH8sSG6A(76iq3Kyq5@j6pEKsYD_7ap=z-W0X514VlV+epRB-FP3<*Npe_kyDerRBU1y#bh1Dw*G%9gBh9XK*i_q8ITl!J}V$cbaWMth^e=xBVpzT?g6Fy-FA49?t;sA;98<7pX&IUXNTsQ@(Sf(p4`v}LF7G6u2g{(H<$f$z_l!ze#_6Yroz#lD@<~YhxZ#cp@@f%X#BP)}|8Mz{D5(3oZO-s`g_WD+M%Kkz85I{6>6b@i9>-zM!*BSP5ZKrL$sD|BHBCT-3T$5YfW`tyIys%PonA(NJ(jB0%F$uzw@axbNP=AWL-?Xva>9I9~nZSL;*z#vd6e20tWN4LgqxBhO&{k$r6sYP2p=jzId-YVC$#+T=PZin5y(qMb(mP92qr}H4?9*)_cNmoW18wGZ<X<W~6>63{!h|F2<O(N>QWcPhEW3lYmcxrmkz7ws0}9`W6UR8~V#AJor|kk7CR81BYlV}vu~k&)fKfejOD@HPk{B(G*BOYY0+M!Cyr?b8c;tGS5iX(;8ao$-L0RYEK9}9Qwk1LHmano&Lz`pt*w*shE={&$ub&lKT=!u}cfjSWo15;3%D|Jsc0Odgs8p}{cwn|gSV#v4KP<!pL*eHzF;}IYJp3QSe5Pd'


import base64
import copy
import json
import math
import zlib

_ACTIONS = json.loads(zlib.decompress(base64.b85decode(SASH_ACTIONS_B85)).decode())

_MODEL_MEAN = (0.7754330004241974, 0.42216157898077045, 4.845066681290341, 0.2690391571999746, 0.649190802764908, 0.5422947717626129, 0.38474963508282034)
_MODEL_SCALE = (0.9129241235454895, 0.4939040192317694, 3.256390917597153, 0.44346035797258343, 0.4421957956795104, 0.27905760741549984, 0.4865360761410145)
_MODEL_INTERCEPT = 1.245742223898873
_MODEL_COEFFICIENTS = (-0.9451280735752147, -0.4680272533738784, -1.4615973335974357, 0.599421907433265, -0.1545610769891183, 0.44222011734216526, -0.025622914463126638)
_MODEL_THRESHOLD = 0.8065185529227787
_SELLABLE = (
    "STRAWBERRY", "MELON", "MILK", "WOOL", "EGG",
    "TOMATO", "CARROT", "WHEAT", "FERTILIZER",
)
_PRODUCT_BY_ANIMAL = {"COW": "MILK", "SHEEP": "WOOL", "GOOSE": "EGG"}
_GLUT_WEIGHT = {
    "STRAWBERRY": 2.0, "MELON": 3.6, "MILK": 2.0, "WOOL": 3.2,
    "EGG": 1.5, "TOMATO": 1.3, "CARROT": 1.0, "WHEAT": 1.0,
    "FERTILIZER": 1.0,
}
_STATE = {0: {}, 1: {}}
_WEED_STATE = {0: {}, 1: {}}
_WEED_REPLAY_STEPS = 8


def _get(value, key, default=None):
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, "get", None)
    if callable(getter):
        return getter(key, default)
    return getattr(value, key, default)


def _copy_action(action):
    action = copy.deepcopy(action or {})
    return {
        "farmer": list(action.get("farmer") or ["PASS"]),
        "hands": [list(order or ["PASS"]) for order in (action.get("hands") or [])],
        "market": [list(order) for order in (action.get("market") or [])],
    }


def _tile_token(tile):
    if tile is None or isinstance(tile, str):
        return tile
    return tuple(
        (key, _get(tile, key))
        for key in (
            "kind", "crop", "animal", "growth", "yield_units",
            "fertilizer_available", "cared_today",
        )
        if _get(tile, key) is not None
    )


def _public_farm_distance(left, right):
    distance = 0
    if tuple(_get(left, "farmer", []) or []) != tuple(_get(right, "farmer", []) or []):
        distance += 2
    left_hands = [tuple(position or ()) for position in (_get(left, "hands", []) or [])]
    right_hands = [tuple(position or ()) for position in (_get(right, "hands", []) or [])]
    distance += 3 * abs(len(left_hands) - len(right_hands))
    distance += sum(a != b for a, b in zip(left_hands, right_hands))
    left_unlocks = set(_get(left, "unlocked_quadrants", []) or [])
    right_unlocks = set(_get(right, "unlocked_quadrants", []) or [])
    distance += 4 * len(left_unlocks.symmetric_difference(right_unlocks))
    left_tiles = _get(left, "tiles", []) or []
    right_tiles = _get(right, "tiles", []) or []
    for y in range(max(len(left_tiles), len(right_tiles))):
        left_row = left_tiles[y] if y < len(left_tiles) else []
        right_row = right_tiles[y] if y < len(right_tiles) else []
        for x in range(max(len(left_row), len(right_row))):
            a = _tile_token(left_row[x]) if x < len(left_row) else "MISSING"
            b = _tile_token(right_row[x]) if x < len(right_row) else "MISSING"
            distance += a != b
    return distance


def _mirror_probability(distance, money_gap, board_streak, step):
    values = (
        math.log1p(max(0.0, distance)),
        float(distance == 0),
        math.log1p(max(0.0, money_gap)),
        float(money_gap <= 5.0),
        min(max(0, board_streak), 96) / 96.0,
        min(max(0, step), 718) / 718.0,
        float(step >= 480),
    )
    logit = _MODEL_INTERCEPT + sum(
        coefficient * ((value - mean) / scale)
        for value, mean, scale, coefficient in zip(
            values, _MODEL_MEAN, _MODEL_SCALE, _MODEL_COEFFICIENTS
        )
    )
    return 1.0 / (1.0 + math.exp(-min(35.0, max(-35.0, logit))))


def _align_hands(action, obs):
    action = _copy_action(action)
    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0
    farms = list(_get(obs, "farms", []) or [])
    farm = farms[seat] if seat < len(farms) else {}
    expected = len(_get(farm, "hands", []) or [])
    hands = list(action.get("hands") or [])
    if len(hands) < expected:
        hands.extend([["PASS"] for _ in range(expected - len(hands))])
    action["hands"] = [list(order or ["PASS"]) for order in hands[:expected]]
    return action


def _tile_at(farm, position):
    try:
        x, y = int(position[0]), int(position[1])
        return (_get(farm, "tiles", []) or [])[y][x]
    except (IndexError, TypeError, ValueError):
        return "LOCKED"


def _trace_actor_action(step, actor):
    trace = _ACTIONS[min(max(int(step), 0), len(_ACTIONS) - 1)] or {}
    if actor == "farmer":
        return list(trace.get("farmer") or ["PASS"])
    hands = trace.get("hands", []) or []
    return list(hands[actor] if actor < len(hands) else ["PASS"])


def _weed_repair_action(obs, action, step):
    """Repair a visible fixed-route PLANT/BUILD slip without touching market."""
    action = _align_hands(action, obs)
    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0
    game = _WEED_STATE[seat]
    if step == 0 or step < game.get("last_step", -1):
        game = {"last_step": step, "active": {}}
        _WEED_STATE[seat] = game
    game["last_step"] = step

    farms = list(_get(obs, "farms", []) or [])
    farm = farms[seat] if seat < len(farms) else {}
    positions = [_get(farm, "farmer"), *list(_get(farm, "hands", []) or [])]
    unit_actions = [action.get("farmer", ["PASS"]), *list(action.get("hands") or [])]
    active = game["active"]

    for actor, transaction in list(active.items()):
        index = 0 if actor == "farmer" else int(actor) + 1
        if index >= len(unit_actions):
            active.pop(actor, None)
            continue
        age = step - transaction["start"]
        if age == 1:
            unit_actions[index] = list(transaction["intended"])
        elif 2 <= age <= 1 + _WEED_REPLAY_STEPS:
            unit_actions[index] = _trace_actor_action(step - 1, actor)
        else:
            active.pop(actor, None)

    for index, (position, intended) in enumerate(zip(positions, unit_actions)):
        actor = "farmer" if index == 0 else index - 1
        if actor in active or not isinstance(intended, list) or not intended:
            continue
        operation = intended[0]
        if operation not in ("BUILD_PASTURE", "PLANT"):
            continue
        tile = _tile_at(farm, position)
        if not isinstance(tile, dict) or tile.get("kind") != "WEED":
            continue
        active[actor] = {"start": step, "intended": list(intended)}
        unit_actions[index] = ["DIG"]

    action["farmer"] = unit_actions[0] if unit_actions else ["PASS"]
    action["hands"] = unit_actions[1:]
    return _align_hands(action, obs)


def _shed_access(size):
    half = size // 2
    return {
        (half - 1, half - 1), (half, half - 1),
        (half - 1, half), (half, half),
    }


def _projected_shed(obs, action):
    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0
    farms = list(_get(obs, "farms", []) or [])
    farm = farms[seat] if seat < len(farms) else {}
    private = _get(obs, "private", {}) or {}
    projected = {
        key: max(0, int(value or 0))
        for key, value in dict(_get(private, "shed", {}) or {}).items()
    }
    inventories = list(_get(private, "inventories", []) or [])
    positions = [_get(farm, "farmer", [0, 0]), *list(_get(farm, "hands", []) or [])]
    actions = [action.get("farmer", ["PASS"]), *list(action.get("hands") or [])]
    tiles = list(_get(farm, "tiles", []) or [])
    access = _shed_access(len(tiles) or 10)
    for index, unit_action in enumerate(actions):
        if index >= len(positions) or index >= len(inventories):
            continue
        position = positions[index]
        if not isinstance(position, (list, tuple)) or len(position) < 2:
            continue
        x, y = int(position[0]), int(position[1])
        if (x, y) not in access or not (0 <= y < len(tiles) and 0 <= x < len(tiles[y])):
            continue
        if tiles[y][x] == "LOCKED" or not isinstance(unit_action, list) or not unit_action:
            continue
        inventory = {
            key: max(0, int(value or 0))
            for key, value in dict(inventories[index] or {}).items()
        }
        if unit_action[0] == "DROP":
            deposits = inventory.items()
        elif unit_action[0] == "PLACE" and len(unit_action) >= 2:
            item = unit_action[1]
            tile = tiles[y][x]
            structure = {"COW": "PASTURE", "SHEEP": "PASTURE", "GOOSE": "COOP"}.get(item)
            if (
                structure is not None and isinstance(tile, dict)
                and tile.get("kind") == structure and "animal" not in tile
            ):
                continue
            try:
                requested = int(unit_action[2]) if len(unit_action) >= 3 else 1
            except (TypeError, ValueError):
                continue
            deposits = ((item, min(max(0, requested), inventory.get(item, 0))),)
        else:
            continue
        for item, quantity in deposits:
            room = max(0, 100 - sum(projected.values()))
            amount = min(max(0, int(quantity or 0)), room)
            if amount:
                projected[item] = projected.get(item, 0) + amount
    return projected


def _opponent_exposure(obs):
    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0
    farms = list(_get(obs, "farms", []) or [])
    opponent = farms[1 - seat] if len(farms) >= 2 else {}
    exposure = {item: 0.0 for item in _SELLABLE}
    for row in (_get(opponent, "tiles", []) or []):
        for tile in row if isinstance(row, list) else [row]:
            if not isinstance(tile, dict):
                continue
            crop = str(tile.get("crop", "")).upper()
            if crop in exposure:
                exposure[crop] += max(1.0, float(tile.get("yield_units", 0) or 0))
            product = _PRODUCT_BY_ANIMAL.get(str(tile.get("animal", "")).upper())
            if product:
                exposure[product] += 1.0 + max(0.0, float(tile.get("yield_units", 0) or 0))
            if tile.get("fertilizer_available", False):
                exposure["FERTILIZER"] += 1.0
    return exposure


def _ranked_sells(obs, action, requested=None):
    shed = _projected_shed(obs, action)
    if requested is None:
        requested = {item: int(shed.get(item, 0) or 0) for item in _SELLABLE}
    prices = _get(_get(obs, "market", {}) or {}, "prices", {}) or {}
    exposure = _opponent_exposure(obs)
    rows = []
    for index, item in enumerate(_SELLABLE):
        quantity = min(max(0, int(requested.get(item, 0) or 0)), max(0, int(shed.get(item, 0) or 0)))
        if quantity <= 0:
            continue
        score = (
            (1.0 + exposure.get(item, 0.0))
            * _GLUT_WEIGHT.get(item, 1.0)
            * max(1.0, float(prices.get(item, 1) or 1))
            * math.log1p(quantity)
        )
        rows.append((score, -index, item, quantity))
    rows.sort(reverse=True)
    return [["SELL", item, quantity] for _, _, item, quantity in rows]


def _front_run_market(obs, action):
    action = _align_hands(action, obs)
    market = [list(order) for order in (action.get("market") or [])]
    requests = {}
    for order in market:
        if len(order) >= 3 and order[0] == "SELL":
            try:
                quantity = max(0, int(order[2]))
            except (TypeError, ValueError):
                quantity = 0
            requests[order[1]] = requests.get(order[1], 0) + quantity
    sells = _ranked_sells(obs, action, requests)
    targeted = {order[1] for order in sells}
    remainder = [
        order for order in market
        if not (len(order) >= 3 and order[0] == "SELL" and order[1] in targeted)
    ]
    action["market"] = (sells + remainder)[:10]
    return action


_TERMINAL_PRODUCTS = set(_SELLABLE)
_TERMINAL_TIE_PRIORITY = {"WOOL": 8, "MELON": 7, "MILK": 6, "STRAWBERRY": 5, "CARROT": 4, "FERTILIZER": 3, "WHEAT": 2, "EGG": 1}

def _best_terminal_item(inventory, prices):
    choices = []
    for item, quantity in dict(inventory or {}).items():
        quantity = int(quantity or 0)
        if item not in _SELLABLE or quantity <= 0:
            continue
        price = int(prices.get(item, 0) or 0)
        choices.append((price * quantity, price, quantity, _TERMINAL_TIE_PRIORITY.get(item, 0), item))
    return max(choices, default=None)


def _terminal_overlay(obs):
    private = _get(obs, "private", {}) or {}
    inventories = list(_get(private, "inventories", []) or [])
    shed = dict(_get(private, "shed", {}) or {})
    prices = _get(_get(obs, "market", {}) or {}, "prices", {}) or {}
    player = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0
    farms = list(_get(obs, "farms", []) or [])
    farm = farms[player] if player < len(farms) else {}
    hand_count = len(_get(farm, "hands", []) or [])
    placed = {}
    farmer = ["PASS"]
    if inventories:
        choice = _best_terminal_item(inventories[0], prices)
        if choice is not None:
            _, _, quantity, _, item = choice
            farmer = ["PLACE", item, quantity]
            placed[item] = quantity
    hands = []
    for index in range(hand_count):
        inventory = inventories[index + 1] if index + 1 < len(inventories) else {}
        choice = _best_terminal_item(inventory, prices)
        if choice is None:
            hands.append(["PASS"])
            continue
        _, _, quantity, _, item = choice
        hands.append(["PLACE", item, quantity])
        placed[item] = placed.get(item, 0) + quantity
    totals = {item: int(shed.get(item, 0) or 0) + int(placed.get(item, 0) or 0) for item in _TERMINAL_PRODUCTS if int(shed.get(item, 0) or 0) + int(placed.get(item, 0) or 0) > 0}
    ordered = sorted(totals, key=lambda item: (int(prices.get(item, 0) or 0) * totals[item], int(prices.get(item, 0) or 0), totals[item], _TERMINAL_TIE_PRIORITY.get(item, 0)), reverse=True)
    return {"farmer": farmer, "hands": hands, "market": [["SELL", item, totals[item]] for item in ordered[:10]]}


def _hybrid_action(obs, action, step):
    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0
    game = _STATE[seat]
    if step == 0 or step < game.get("last_step", -1):
        game = {
            "board_streak": 0, "latched": False, "divergence": 0,
            "last_step": step, "mode": None,
        }
        _STATE[seat] = game
    game["last_step"] = step
    farms = list(_get(obs, "farms", []) or [])
    distance = _public_farm_distance(farms[0], farms[1]) if len(farms) >= 2 else math.inf
    money_gap = (
        abs(float(_get(farms[0], "money", 0) or 0) - float(_get(farms[1], "money", 0) or 0))
        if len(farms) >= 2 else math.inf
    )
    game["board_streak"] = game["board_streak"] + 1 if distance <= 2 else 0
    probability = _mirror_probability(distance, money_gap, game["board_streak"], step)
    selected = probability >= _MODEL_THRESHOLD
    if selected and not game["latched"]:
        game["latched"] = True
    if game["latched"]:
        game["divergence"] = 0 if distance <= 2 else game["divergence"] + 1
        if game["divergence"] >= 8:
            game["latched"] = False
            game["divergence"] = 0
    if game["mode"] is None and step >= 48:
        game["mode"] = "mirror" if selected else "open"
    active = step >= 24 and (
        game["latched"] if game["mode"] == "mirror" else game["mode"] == "open"
    )
    if active:
        action = _front_run_market(obs, action)
    if step >= 716:
        return _terminal_overlay(obs)
    return _align_hands(action, obs)


def agent(obs):
    """Kaggle one-argument entry point."""
    try:
        step = min(max(0, int(_get(obs, "step", 0) or 0)), len(_ACTIONS) - 1)
        action = _weed_repair_action(obs, _copy_action(_ACTIONS[step]), step)
        return _hybrid_action(obs, action, step)
    except Exception:
        seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0
        farms = list(_get(obs, "farms", []) or [])
        farm = farms[seat] if seat < len(farms) else {}
        return {
            "farmer": ["PASS"],
            "hands": [["PASS"] for _ in (_get(farm, "hands", []) or [])],
            "market": [],
        }


def _kaggle_submission_entrypoint(obs):
    return agent(obs)
'''

AGENT_SOURCE = 'V22_SOURCE = r\'\'\'"""v22 price-impact route agent for Kaggriculture.\n\nThe production route is a real fit-only trajectory.  Runtime adaptation is\nlimited to identity-free weed recovery and in-place ranking of existing SELL\nslots by the official 1.32.4 price-impact curve.  Ordinary turns never create,\ndelete, or resize a market order.\n"""\nimport base64\nimport copy\nimport json\nimport math\nimport zlib\n\n\n_ACTIONS = json.loads(zlib.decompress(base64.b85decode(\n    (\n    \'c-rk<U2hyoa{MoR<^$)06zMmvG-nCN6$MJV!FfR}7VsGcjPt|VZ^r$1Yei0XPiJIgWLEVowYs+fIn!O0Rb8DK85#N0|DFBEFTei%\'\n    \'Z@-@X%TH$?Za;oJdptk;&tLxg-~Z>!AHID2`!B!#*Wdp4%jciY-oAU-efh8U;fK$E{rUF&yB}}w&d$$1zTNFUoSm=DKVIMOCx8C9\'\n    \'+r9bn$Nk&g?WeQzSF=C=xVyW5e|ElFKR*8B{AkqgUjO;?hso83@&9zT-+lb_bv*CyA3nYP`ssO+liy8;_w<9~iT^f;4-\'\n    \'fZmKEM7n4$lnZhtKcs-\'\n    \'u(RK>YqM;wZUW*<IUMHh6|71n~q~X>$}_6yXQ&kH#7evcX+nl<kIsg!dtjsBDW&88&>ekgx`<$KQ`gBEuI##(SC>fJngryd*b%f?\'\n    \'&0{3fBJ1tPDl0p?UXsk>yDE=+~D=?v+<t4)X8|`q=q{U-\'\n    \'?3XioPk{vU@N;JW<TSrbaVsJdUiv^W;|RkX})16G?<TWwP7ddTWx+_(Q4yP=wdAVpw1^8S#ADK60J7*lWsP1t4`JebMP%<{xx~H3\'\n    \'dRBk@ogkLkYp<6Lnjm25025gjeBORZsR`waQe$WpCyii2mPFl>uwESNL|nPP0t5t(>3Nt>+dy=f_=?34ldOnVluni_J!#&j@S2hc\'\n    \'e~f0e*V+$;nTajcmH<$@~T|%<NfFMW$HhyHxKt8mVKH&?(TjI-\'\n    \'6lgGBe+F2M0f(N8n5?ao;YUs^3KWZ+g>*TF>P`;sTf0HbvY^!M~?HAo?d2j*7fVn&$pxNp%pM57BuPja5$D)J%$0wI1u3fTA!}r-\'\n    \'qxt26K0Ltb=pn#kByKx9CHvsY=q3MNkCUx`(D!qVas<qZ*Y<<G;uc~>U8h969A_>e0ciya<~2tX3byZN-VsX4alwkm?kKM_Am9$e\'\n    \'Xsv5U2W#yZZrPvR`qYWqr1h~G{v)0l48#nQ&30dK!ID#Z!bhjDOWXl%QkZzWsz#${v0LkZ7UQ2F}HH|zsfDG5$%jXP7)rpRVSX@u\'\n    \'`tEVn~c5I>o+7e4I%g*u$y?lmZ)&k@ui)3ktGIX(39U`3@^?oAhG$~+XNiC|1?Uktk#Ph!Z)25T<WrNuF&(7owt7%5Bkz0p7rsdr\'\n    \'vX}TADZ&G7ILBorc09(O{h){QvqIe3?VLOJv+&fCZsq>32>ZI#~M&_!5x(17Gb)Z#Q^f*e{XLe{-Qd<8v!YOcK-\'\n    \'R)byCeRJa`Wj&(1e@m3L@|KaNGAv{}<ke~$xZ%orqhLB5o^%wR4l$`inFW~BN0XY#G%pQfL{rK3R;C>zlTaVAS(W)TYJ;eOLaw=;\'\n    \'oHUuFdeqtHvx$<Lm(;?Wae4LPn^Y`f3{HxmOp;+fJh{4cj9h6#Wx&*v1L(lA?}oHO11&T_^xPb0F2SIng|uZ>^MfN@rXoSFxv;4l\'\n    \'?{#MG<c=_09tp(RQUGmIMC*f}-MG#OI(i3j?n24>S*;lPqMuNcs+hX{x3Vqy&kV{*bC0@ZRfS3~CS<tHtn{bhS-\'\n    \'4}X2tH+0|hkJwKZr*}(n*;mA(L3tkou?bb|!^Vva)RBxvkdw}gpj@EXm7S5JVc8Wt-L&J?-\'\n    \'Xt<7cG?U?($A1hE(xqWj2To!A<iX27e5Ls4p8A5HL<s{{)-\'\n    \')JGF;1`4)%gdgs#VR&E~MSCM&mZ9XPqRVi!Buh%{U^%iKM=3}f8;`@63d^Ekx4n*6=ue%;-\'\n    \'^f0Hlvj{~=PJwNt$kh>tdi8r&v&&N*>xBDM<4-bDmJHL+K$g~XJ&;Bfzwwbe##`D*gI7o^CcwsEQ-gq=-\'\n    \'?x9$EyfXOTVIV^WWcHn<t?gBv`@mNC+F^DdE<Vr9p)AB?kGHM?D0e~c>b|sD;J^S&fTYP_iG_LO$k0avVVikAgHEFqBN<OV<M?C?\'\n    \'Hc|^W+ibdF9fuF|1Zt3lX>iKrr3DpA=m-?R^(|rfCYID%4IRYUg-\'\n    \'i^NVIrfOg{=w3EUfD#VRsahG43j(Povi>1mlUp0tcSFRHx<}M-\'\n    \'(3=9j%9UerVBo%+P79gpg>mcSXKe&wXdd)<i|CnWIToLY*P0wbJj#sFiZ)bs`SiRf(y_h0vza$~F;BO^<W<f)|Cz8u0nmMf}DXrL\'\n    \'}CCKOT7UlMn@5KZ==C=T5^MfIr$cvYSUeKtOjIhE2a!;m2i`*2t&L-\'\n    \'w?aGBLl|PWsqcHL2zT&)s7}`U&L`ykpUJ<BE>Mzx_~^l(1LUCGo&X)hL$r&7ulCHk8m0C;SRXdK856~iF>o-\'\n    \'?lxvQb+IWxv&{pK7*7uI%8@|}tB$mj9da+hs!u#6pN*z5D}`YiAC9o_oJR+pn69V*&XkstEMN<=Y%FCF7s#y|Ay7C)!J^v#wWXoB\'\n    \'CLdztRz@HvDC~0ud7YBzXJuG0g!Jz!)mwJW;4q506jC+k!IC#-V+o<>xs({cuDdhg&7ZCf!>1qL-TfIbdEhzSZ$Kohrfk=ok)rUF\'\n    \'q$0@x9gF4Dh0ibaO(|&QqTbV>^_7#4T;+q=wHnN>nu6J<4>baq6;x=6Vds}iy&Gp*#W=pog{;No;jBg}0Q&olgk&0QKID+#(_&^P\'\n    \'XEI)?wp75K(qkn$-\'\n    \'~x5oE*n`$KT+dHB?Z#94aH;ofTA%uRIcEY^wvCzB#T<{vA~Y7OBkjy#XNJ~<Ixh9nFZF95iAEA?m|@6;^S$m_uA|)qGxcIdl(zxb\'\n    \'AnH543b5JuZ7(dZnDr>ON|ivw}_+es-\'\n    \'Z8jdJ#M(EmP;UY|>!qH3;JOFE@r6v2VMa;72brwMfI9Y7c*(R%w>;&&G&KG|jReN|d)jf4tNKJ`8ZLGm)R=UgGp6NR2GS4rPw!Fb\'\n    \'bN^Y#5HB1j4eDeW|mOh=q-k?O!+zxx7-\'\n    \'=*&i%oS4!qwR=I+l&uiDpN=Nuvk0lDEAW#nWx|l)iNmWG#1fC>XTVIT$f#V2#C^|5DTBQhkn|}IbeBo_a3%@Vz&jbaM#b6X<V~9^\'\n    \'*3aMNWgH9N503c3XSMH(sK(WjtJfyk~QWX+GpDeEgWW^U7CqKZ)kpLfoVNvrjUg>4qnvMw?Vvy9Gec;1QJ~jal10h7?VTH%d(quz\'\n    \'B)bl7T6eWlv{FH06#CtY=sKLIH7j&+YBUx`+0r!+eQ{I&J=23z#_vc|MEawL%aQIWAawe>Z?FP|ExLK0E+p|{k*?OA`Csw^FB}KI\'\n    \'o$*99I0Gkl0m)9P`|Dt2CN>`qyc-LXmu5Cd2o!SUN>OzDV@*3+rW<hAgD?;=UTjm9@iL;1$p|E*c%;VQQvPe~$5So8wMuO;r<t77\'\n    \'b(uHBcTY4h;_KMiPAfF!BvjzMoFfYFtu8;UUr8Cm{J0zsZS_lW3V8=F+o8BZ#GAtykeAa2yG$0`v`_X2mq7SrK?pnM$-\'\n    \'RiSbCt9_TfNotPdP3FeoipHvk8_bHdfr%D&Wevel&c4bQfUJ?0lZZJea6gnmC8$PxKJf3Ae^ck41GgWvUhmE`|wJq5ML9PJTI2zv\'\n    \'S<Ux={f<ZuN`z*9Y!6J==9)O*NHS4t=L!Y^j|$z3H;H6;q`>i5F{w6TC0y}FvWZZj=QZs3=;i9Cvp+{##mZrL)dzWaF+^NS(G_5&\'\n    \'MAUy6$1No&H$~^%)k>g(OT6xLX0rx0XLmi9#>)&UkW)E(rDTdd&?g)nlWZ@qii0%$X<Fb;<l%9VOwtElmJ31UZ`$(expkjBMXp>0\'\n    \'QjHVtDOg)>d2b<kkFP`M2F!}7IV%(^P~Eci!N#n=^~cPB6w>VMkO`8)wWW(FCkjJsZ1918i#>u&E+DXDk5QDk|15`lPCIsv21ZHk\'\n    \'*GjjT31#kf+Jl;ht4Y#dXG%!p@nplSvk9p0!~W{)^u4GLCASyvE>_})GHlQ()81{@+s0-\'\n    \'MhCr8^4CoHq^6m+)=dX}a;Ab}!w1k1s*}~2CW22S&8bZACoHjyxkrb|WMrCuS?_Mlv@+iw;eW+a`r&=xGYkNpr9Wn?rZ`d=RKngZ\'\n    \'BH4Va#yP?UA!<ncQXYwrsE{XnsF)=2`8R#H1Qk}`FRP?>0I}AL6vPy1NXgK3On9oa=1E6{Yz?W+(!3rU;LVPYZ7%3bM|YcibhnC|\'\n    \'(e#ij$Od&;dP$Si8YM>eB2lpL&1M~2@2{)?2urA#?_PgDNcJ~x0c!ost5LB&EcUj0!-J-\'\n    \'4w%1DXiT_n?VoKB5<WDX`p;?BC4kXEa$n<w1x23w@J6Q=q($QzkqQ>d70(7*s089)Fz_gxAa$9~WN2qdGH~=UegFkS@W`&-\'\n    \'a=ki!61F$_lU|R0|<w0m+fYwyO?X%FMGmoSxgBwm4Mx#>^NJD5{jBx=o0g8eq^UEQ*9yXJe&x%zj3E@E)43Z@Gbc5xCpT-_fTP@3\'\n    \'V<E6B0c~DJ~T*w-\'\n    \'ARBKzub^L8&Y|}t%?(#3zfzmCz?^9S$YjkJPM4!Y{oH@xDbzQ^ly$bn0X(Uz+KyBKwz@64n@ZDRcAET06Km9OIdM;N>O`?%hE0#_\'\n    \'jcbF-<(xPcW1an>EUD4E71pp`3bB#Aef-uWuGlC$!A*p!vm{Go_<+bsOB!(fCr@Fc#%W*bQ{Hkh$Xg8J0TJ1giwIbCy^rc8FEVl-\'\n    \'4B*sep>in(iZJl!dIR70Tp(qZF0JX?ufbdxWPC~f>vEZU-\'\n    \'7+^}3tx^AoIgaI$9b(!e&%uM^ZO)A#pjuTaPUH|;Qz$1IW^1NJ>SP;`*%nX8sJRISVfR^adROL!WRS%rlRK&hNuje-\'\n    \'<ba4{LziVIgP`qsq7JpjCrBr;SpgqQitoDToU*2*<x3s2Q=Xnu@RU<6q|ls!LNd+P@Ekfzr}&AuzG<FxE{wJ`9oRyWK$k9k#EvBJ\'\n    \'h_GT8TBejpVqrL+3@=emNUe<<m(3>Hg)2xiznaQCu(4&7g*|3A2|x&JV@NZUnu8@3qO#jXPxQ&kDAH_}P-\'\n    \'K@jK`xjF4{wjDaK*%ErGelZDTrI3F|f?HZ4f+bAX_%Kwkj6ft=2RbdHV9YwV=i={bt)LB))y@lC@lig>OqMiPKp4sA=F~=+ljG=}\'\n    \'d3>oy~GdQHx4gZz(XKtuSrQ?Gce1h_*>2KsDP(mKs1ecBl#=v%?xXry&{7@XdJ?IQ)6}tSwbcra(np%GFWzIrM=L-\'\n    \'XxHemrp(7s29W@$Gc(rGjT&?#!G6!Y-\'\n    \'J7b>xs&j>AcFvoC}!vC#tl!ZxH(5B&z!`<&QBYk5?Z4y|vPh^r@~PXjSx!fQ7jh2nhzU13ZB)7Ty!6rwKD<e?~Y77F1?SLj|I-\'\n    \'3;<E^DQ1(}*t#5WK~N6zjHYX7ewAyQr7TM<TL(rNb{O6y&qX>g)@(SL`n5)4T#C>(zLHtSHgd~|2p>TrRuYN2r`GpT=^uzxmo5q1\'\n    \'c1oA8Ut+qPe}XXdGAVH<u8i}{y|bg{?VD{VJVCh7oK_2@!kJq|6R3UC-Y#v<?KefKu_s7#2i-\'\n    \'rk_^6WC*4|6Br%x0YcS>WE9z2fRaw3HZJaVO84UZt~+_P(F!AvVEcQk22%ZO60AY-\'\n    \'~~y~+yKO1{?#rouhcDf=1XKeW2CvZfW?bu?a_A*<{=sqZov3wq6~)((B{tJv`fB1wK|h2zh)DAF^uT8%zlXCkzk4`Os8F=w`s<|6\'\n    \'FHn0><Y$xOXRlrr*`>QMPDX2xOj4M~~tX{#M8>2lQx`h^ojpb|V|!IRq3AvwZCns^=y(+li+3l`V2_O;yaQM;yC7ATz!o?hQmTzv\'\n    \')h?g^Oc<r35d`RQ_d>s;bD&SjR9*KIYgaSC`XrHNa6M3-\'\n    \'tnlaV|{zoM!4F4f(+{c+<q*J52n=M*&SM@*q0%Bftay}3j<2vjM_E0l_jEL3%Db__~VDpDQ3b9|y4H?QjeX>B<&^U@=I8watPNAO\'\n    \'^>!~yeI-F#)IX@DSwoUyzl!;AhH19<U80sDu^n4xWHvmy?SL7-\'\n    \'VAN_lDD5xFa`Smq(nJp<WKZBljZD|9#<n;+DAdeW}#nl$N><5)7uZ9dT#7m=ELg5NQ#78tsE>uqY&kXuw3EX(^+w4c=|p*DA!AaT\'\n    \';A=D47EEoIN)MoTmu6aA#rl-zj6E<hA-Q<+Fv(JH@lTC?@wp(>T}2en<fJ(m!+U=5sza`Nz@wsEF_#&$8)<oLjwerP^*(%gw-\'\n    \'D`W}%O5MRf`4e%8{9+Ie%x8<r1KHxV2DE@=x9n3+;&1xOY#DK;`3u~uSxyFdXD-\'\n    \'w+ov}8Kxz(UB1k)u>2h(JeSiw5b3jIuPmL|wXY6OMz_7VQ%B2k^1eX}5%>ffbUNoC%8lW3ikRmD+pep&h!jR5UCO^JZc8GNe*IW_\'\n    \'6$NoY3=nc<>nyoD1q5LXL1bZU8A$FNvWa9%OaBH!Sbr4_m|aqw!J==5zNc0Ce~j|aMPei!kuF8JE^OnFsrVoqO3k+uT<D<jA$&ca\'\n    \'r}Ktqa~TY<p&Cr!<cybL>{xA$%K1K_!6?XU%jaz=pSe0}`3ebUt&?*%JL`vFwiWV@vZtM)at>=ReaD8c$GsfUDf(M25#3NN}mq|j\'\n    \'@+0Up;<Z@XA?Bg+h8IMCBKDP2^C0LhTOZKFePS6!J&t1H}-y;-\'\n    \'Kht1LQ&w&wc8V3*Hf;l(&tD#M48qs$d0>~B>tNBK=d#2rKuS5N}9X{Lcy$x%e9QfPqPr>?9r6T6EoxyWI*0w5@^G_SYOt0Q!bCB@\'\n    \'6BP%r`N#zHZ#K42Xy3$vRnmObYJtNSk9inoak=Wz=nAQF1d$CuFA(2`ncLvSB?MV#-\'\n    \'mQv@d|4wcnz?b@P%?O=+iv62P0bQb<tPU|0bod6^_E*$FpBn~1)Srjd^q{>N(qN~xe+OZQIkz9!%5m2EOU@R)piemYLNV*jZ8ajI\'\n    \'YSF}RMzl-17wYI0|tsK<L*d5?bvD~jyFC=6_MigI!NWw07<x#=qM2RK<5@Ibdx41moJbSGS9blPN!?QDmnmcCBLhz%YffK-\'\n    \'}8fQw7j1C7sH&KtuUeDo|I9?$TmHaw|BW(?-UZEb)WQpDZXH7!KwXB#^STu`Ac@H!J-\'\n    \'DF@aqme{bgYe&MX=zbeR!05w)l%~n+s8h=FsGOHiX0Cy^hFN_+vQ0ZZ0p~h!B(!)TZb8jWEEV;t~#`wQe%Y)f9lp!$HE<4N`R}wV\'\n    \'>O`#TI>J~R8ba&Zkx_McEAn@c^}$9TY7~cN5*ag)tpK%_1dxrf{GBe;wFs{NhJzq#4>1Exq){ifP?>^*c0ipC7HEEn%9SrOllL0(\'\n    \'I}ftOdJTlLHPsv_C}AjBOxqV-UA#o4RWcgMXc2uv9{$WFe*(h%hb@a)+nBg9zh}&l#)~?T|yhgmN*$j9D7*gq#wtW*3XDASu>kv0\'\n    \'e~aTpJVtugjIV|HGnqJ1i)sO#71+Vvy-A~Bq1xLd$|p$e%#&Nzeh~SbVU?A;8``ht?&r-\'\n    \'oT4~a5LXKQNn2h9U%&Qi^R~VCPwxN1v2Rv(-PY-45^A&~r~tq&YWm;xu2U{QDha}w{QxwP*nXk}a<epbj^&o{aEF;Fjtjw};-\'\n    \'zY$Zf!Ah%Sq8L$LZRfgwwTonefjl!E*7LORPInFPhe6Cv-^Am-hejVrg8o9*&qL&*MsLqdvSYsVYwKRnsi4R2jl!7vu`e(Q$TJ{S\'\n    \'#a_KW*Q2+xb*2k0GLU&h=}}YDHO_$EGgS$s$t9UZP=cjlP!QG|@iQOv2ZLKvdb}242(UV_V~EDDwtmNviimIzjc>rLTkv3#}Qotn\'\n    \'!`aK*j5Lo!F<+I;tQ`yt+PI!WR(E=K3+w=!tV~qQX@A$z{}q%))BV?uKr`@hK8Ncr_ZGcHT*p5GgMsfUu=T2|*Eh;tXe{O#$g6ju\'\n    \'ufra;XhD+)1hg9D5e*L)Jw#&D+~yy%$q=qz;FlywD#Mg}K5{U_YG_&6nXZG`%@PF|X6IW-{Uke}oel-\'\n    \'m0^?3B?MlUaD1klP{eaMc$}TMK1N|NfbBjeA1dpy5N}|e<eYXTbFP-xHqIXmP<~PB^5*!`cSUy`O=#UtTaD+_#NWdMF;fo@+2r>X\'\n    \'PGk*@NAbq)5u|Ebg&8e1Da<P5b^=8Y~^1QQ_L3s1)Zeo6^c#Ufrz4qJcPD!<~j_9l4cAg$SS#)*kOSKV(}VmUs4*GVH#*mF~ksz{\'\n    \'ind8XC+i}=*oy=bPKYFQs(4&Qz&=$mD^DTZV(W4)x?G3D&KhB!DIC1-\'\n    \'M@OCBo~(fbJJrTQp~ICN!v9}C?rPd<_xJ*?F+57QLNj)@(MhxKYRWXd`JyYsPJT2t7~su0W~JrB4Y5>rHwf=|6z&3nc)M;=UElyq\'\n    \'a9#~&4rf*=E5;f(8z+R_>2$c8n1*<b5u~YXZVz;Ma76=r`1je%&ZDC)sp?XDD(@eGyw_)76b^O;nS$SCEe<#g{j7r+9D!+-\'\n    \'+p_sS9v%d!x#BTfHXc2-(oETp1;UqT9$3<cf-\'\n    \'V!+M$)zT2<R(e6!`0x4Lx=B;Cd!=&*JFbJ<GUTJuhYCso(7#ZYP5*7`+gQjsv9b(HEt%W9j-\'\n    \'rlPmJ&y^a`?m)_3?!&@@I1p9I(mE4Tku<K_+pMcadi^n$53NiT6u_5P)=ITflvz7u2M1MWX{B#jHB0N2zR$H*r_G!hA5fj|s*PsG\'\n    \'fYvCsleGASjk<2)CMB9wBbuY6z>!o)s}bt7ng#Y?rn&**)n%x-\'\n    \'((_#<(dDGIoMICc0zNzTVyg#fzs7rzg%gp*bL^N3{HbP)ov_zpn{3Dqn~CD!Kr4pMj9DOYsID$DWI<<Q43y+&ke0L{zpS$cJ;nvE\'\n    \'u=NU<0wl6lW~-\'\n    \'w}(T*!HXy~8XG`)DHAS$aRWtutFQm*n<VbmPXKtT;Id^bsg`^qAfO5l`diO|lI(qsDwQck3_=6~YhK@2vBbq3KkJN}(rR+4*^gxC\'\n    \'xtslCr`F6RL1%LACy8bR!;5bou5UtRw-\'\n    \'CwvX>P}9+;10Z?z`&EQ)7`rt;Hw9ExIvW})RC6(DLvzeBIZVPA)5U_zvBCf_XmOo9D@-ZbuMvY-JljAI6S9XjnwYd9g25#kBV~N(\'\n    \'RxFv5CrqHq+{!+CgXV_=7B`8^<T|N@Y_%l+PKrH)q$LFfT!bYxU<P{f9Zp?cyiCJp(sg>NGB_Bs-\'\n    \'QOW6h_z%QTU_5Js1kE0lE`9#!h=zn5S3UONIZ%wp}aIW$`aKHI$_S%AGpw9ZkOf!YY(50o2;1t11)i4MyP~tC-Tb7j@qk~(Op#S(\'\n    \'?dm^9z1ars?I?lxhzY)*JEYoAR(F^Itm%7g}0{GtE$3Eq`U?mRY=_L;+~7mZ4=S>C<%;l)nax|*d~0&v8h3Il+Pxso`T;NSZr72)\'\n    \'|TOZlVgRXf?Oxop>4m|PoG`nLtA2Yfh}g|Aq&p$eWg7160Z##V@6wp25q{6Qrk6#n;dq~g3P$fmxEB9UbCz>`R$t&DqDqot8Yq&E\'\n    \'Ikt+L|967OrpEh=dldija5W4sv8&lNzieY+Exq_mEN6HnoDoJw<RxtlE4^meMlF`(obT@iOp@J^W0dqWgI_-CE0R8Xei#NHig^@<\'\n    \'S)K1QomJ^<dsHu()w_b(vzU?N7jV#xtu-\'\n    \'>T}8zRkqyZZhEn!r^{v$RbI?}Ujy9B(q&3DEY8ipLR96p1bxJ^sa=wKSrX&=srZ2_{5!vg9@vyYb#lBVQT!<S=3se)E97uN%VXIu\'\n    \'kO6__`>I0fc0Gf=)*l7W{jAdg<x+pE_+sA#VQcpR{ywE<I{28W2CbaIPhwxk2d55ybG;LW%Q7WuMpshy-JRJGq+l9_Hq*HBTidb0\'\n    \'cDbNAo+e}ww0Z?+LU8LIR#5TOFwE>VjBmh&$jUo@>>9vDit-\'\n    \'7LeHwKh@g*yv?x{pSD?)v4l4(Lx5Tc1QGtSl$rkrxKJR2du&Dx7u3<O&$cP?_GL(p0J0Y}6DZj#paVU&}W3))-\'\n    \'^YgQ%X>HeALU3zFAtlTc(W6=XGasDz;!hslCEvxK;INh%wjES89o3#ou`O32wuNv9<=RGz0|2QVxela%HH+->VYFdr5j2g-\'\n    \'p`Wpcz|sfm2kU^x;+3CaRceejTd%}dz<G&t%mTLXBQ#rFiGYFq(IrRCJfu#M|3r4gNDTltSGIkKs3HN&@w*=ewOfIrc_&c?2X*<5\'\n    \'QP>R`H0ky@x|GwWPYCAr01JiP%dVV@`C)3E*M`XY`40=FU|y1lRR_F#k`vUT=JWqaF!(^7OY{UX~{!nQUp-\'\n    \'_}(ZvNep50cTGDBJ7qYV;UX`$k-\'\n    \'JsLRq8~6%X3^rKI5MLU(q`u60y#fMG82(F@*A<B}g<l#5MX6v$5@VVjjwH0h#pTdb(xQSR&y!AZ;GI?c6?oe5>hF)oTxDJ4EMy>k\'\n    \'Mwd!<QOeT;=j*4k>+KbQ}mm9%H%vQC9MvBEm4dX*96m+BO)G_^xjq1t3(ooE9blOtLYDgq__0p~ckXL)?)Z(yyhrlJN{<kMKw@yv\'\n    \'l<phq>%vRI>$wXpj)4AEp@LlXTcwXWnWQ;{7R)xwoF1TqdP+6vI+=n$cdi0r1=OS;ezqDZ#_Z|qS+dXQNo3{7nbrWNUSFf^J(0E4\'\n    \'Q9rJ0=!FhRLENyPrR&OV!SV?RO}R1-C|0GopmQJ<sIb4Z+4h!1cpDMCe5B|}r^*ZUag;6-zVDS<pyO?Zi6oKxnLJNU7-\'\n    \'072m*m>$QcXh%MSHylefSvJg7)0T|@F@vW>XsAxR0)!|2o3f`}u=EySSlG@t;BmTaf?w>WO|;-\'\n    \'*TC#@zB7zhaaFjrYwze64TcpNWkV<Go;P{56SvM>C?kllLrp-sVA#;{RMM%X?t5Tc$@NK8cB_{&ajCt|Rh=bZAQOGrTGTd05h7t(\'\n    \'e>aBX2rccNorthpCSP92%%Wpl_QQoEuiQCL8?IgZbkg5_{$(&x!U&M-\'\n    \'E3qmT8M53(<quegI1<6C&WeSr~H?Fh*pi0Mu9e!HO8+I;Bcp1Al%O&efQ=Y&+L`I`Fwr~CVHV66mkH`N9*sTax\'\n    )\n)).decode("utf-8"))\n_PRICE_FLOOR = 1\n_MARKET_PARAMS = {\n    "WHEAT": (25, 10000, 400, "sqrt", 0.8, "log", 0.2),\n    "CARROT": (35, 10000, 450, "log", 0.2, "sqrt", 0.7),\n    "TOMATO": (60, 10000, 200, "linear", 0.4, "sqrt", 0.6),\n    "STRAWBERRY": (120, 10000, 100, "sqrt", 0.7, "linear", 1.6),\n    "MELON": (250, 10000, 300, "log", 0.2, "sq", 3.6),\n    "EGG": (50, 10000, 332, "linear", 0.4, "log", 0.2),\n    "MILK": (160, 10000, 122, "sqrt", 0.6, "linear", 1.6),\n    "WOOL": (200, 10000, 105, "log", 0.2, "sq", 3.2),\n    "FERTILIZER": (100, 10000, 200, "linear", 0.4, "linear", 0.4),\n}\n_WEED_STATE = {0: {}, 1: {}}\n_WEED_REPLAY_STEPS = 8\n\n\ndef _get(value, key, default=None):\n    if isinstance(value, dict):\n        return value.get(key, default)\n    getter = getattr(value, "get", None)\n    if callable(getter):\n        return getter(key, default)\n    return getattr(value, key, default)\n\n\ndef _copy_action(action):\n    action = copy.deepcopy(action or {})\n    return {\n        "farmer": list(action.get("farmer") or ["PASS"]),\n        "hands": [list(order or ["PASS"]) for order in (action.get("hands") or [])],\n        "market": [list(order) for order in (action.get("market") or [])],\n    }\n\n\ndef _seat(obs):\n    return 1 if int(_get(obs, "player", 0) or 0) == 1 else 0\n\n\ndef _farm(obs, seat):\n    farms = list(_get(obs, "farms", []) or [])\n    return farms[seat] if seat < len(farms) else {}\n\n\ndef _align_hands(action, obs):\n    action = _copy_action(action)\n    expected = len(_get(_farm(obs, _seat(obs)), "hands", []) or [])\n    hands = list(action.get("hands") or [])\n    if len(hands) < expected:\n        hands.extend([["PASS"] for _ in range(expected - len(hands))])\n    action["hands"] = [list(order or ["PASS"]) for order in hands[:expected]]\n    return action\n\n\ndef _tile_at(farm, position):\n    try:\n        x, y = int(position[0]), int(position[1])\n        return (_get(farm, "tiles", []) or [])[y][x]\n    except (IndexError, TypeError, ValueError):\n        return "LOCKED"\n\n\ndef _trace_actor_action(actions, step, actor):\n    trace = actions[min(max(int(step), 0), len(actions) - 1)] or {}\n    if actor == "farmer":\n        return list(trace.get("farmer") or ["PASS"])\n    hands = trace.get("hands", []) or []\n    return list(hands[actor] if actor < len(hands) else ["PASS"])\n\n\ndef _weed_repair_action(obs, action, actions, step):\n    action = _align_hands(action, obs)\n    seat = _seat(obs)\n    game = _WEED_STATE[seat]\n    if step == 0 or step < game.get("last_step", -1):\n        game = {"last_step": step, "active": {}}\n        _WEED_STATE[seat] = game\n    game["last_step"] = step\n    farm = _farm(obs, seat)\n    positions = [_get(farm, "farmer"), *list(_get(farm, "hands", []) or [])]\n    unit_actions = [action.get("farmer", ["PASS"]), *list(action.get("hands") or [])]\n    active = game["active"]\n\n    for actor, transaction in list(active.items()):\n        index = 0 if actor == "farmer" else int(actor) + 1\n        if index >= len(unit_actions):\n            active.pop(actor, None)\n            continue\n        age = step - transaction["start"]\n        if age == 1:\n            unit_actions[index] = list(transaction["intended"])\n        elif 2 <= age <= 1 + _WEED_REPLAY_STEPS:\n            unit_actions[index] = _trace_actor_action(actions, step - 1, actor)\n        else:\n            active.pop(actor, None)\n\n    for index, (position, intended) in enumerate(zip(positions, unit_actions)):\n        actor = "farmer" if index == 0 else index - 1\n        if actor in active or not isinstance(intended, list) or not intended:\n            continue\n        if intended[0] not in ("BUILD_PASTURE", "PLANT"):\n            continue\n        tile = _tile_at(farm, position)\n        if not isinstance(tile, dict) or tile.get("kind") != "WEED":\n            continue\n        active[actor] = {"start": step, "intended": list(intended)}\n        unit_actions[index] = ["DIG"]\n\n    action["farmer"] = unit_actions[0] if unit_actions else ["PASS"]\n    action["hands"] = unit_actions[1:]\n    return _align_hands(action, obs)\n\n\ndef _shape(name, value):\n    value = max(0.0, float(value))\n    if name == "linear":\n        return value\n    if name == "sq":\n        return value * value\n    if name == "sqrt":\n        return math.sqrt(value)\n    if name == "log":\n        return math.log1p(value)\n    if name == "log10":\n        return math.log10(1.0 + value)\n    raise ValueError(name)\n\n\ndef _market_price(item, inventory):\n    base, equilibrium, scale, below_func, below_target, above_func, above_target = (\n        _MARKET_PARAMS[item]\n    )\n    if inventory < equilibrium:\n        amplitude = below_target * base / _shape(below_func, scale)\n        price = base + amplitude * _shape(below_func, equilibrium - inventory)\n    else:\n        amplitude = above_target * base / _shape(above_func, scale)\n        price = base - amplitude * _shape(above_func, inventory - equilibrium)\n    return max(_PRICE_FLOOR, int(round(price)))\n\n\ndef _is_sell(order):\n    return (\n        isinstance(order, (list, tuple))\n        and len(order) >= 3\n        and order[0] == "SELL"\n        and order[1] in _MARKET_PARAMS\n    )\n\n\ndef _impact_score(obs, order):\n    if not _is_sell(order):\n        return float("-inf")\n    item = str(order[1])\n    try:\n        quantity = max(0, int(order[2]))\n    except (TypeError, ValueError):\n        return 0.0\n    market = _get(obs, "market", {}) or {}\n    inventory = _get(market, "inventory", {}) or {}\n    prices = _get(market, "prices", {}) or {}\n    current_inventory = int(_get(inventory, item, 10000) or 0)\n    current_quote = float(\n        _get(prices, item, _market_price(item, current_inventory)) or 0\n    )\n    later_quote = float(_market_price(item, current_inventory + quantity))\n    return float(quantity) * max(0.0, current_quote - later_quote)\n\n\ndef _impact_slots(obs, action):\n    action = _copy_action(action)\n    market = list(action.get("market") or [])\n    rows = [\n        (_impact_score(obs, order), -index, list(order))\n        for index, order in enumerate(market)\n        if _is_sell(order)\n    ]\n    if len(rows) < 2:\n        return action\n    rows.sort(reverse=True)\n    ranked = iter(row[2] for row in rows)\n    action["market"] = [next(ranked) if _is_sell(order) else order for order in market]\n    return action\n\n\ndef agent(obs):\n    try:\n        step = min(max(0, int(_get(obs, "step", 0) or 0)), len(_ACTIONS) - 1)\n        action = _weed_repair_action(obs, _copy_action(_ACTIONS[step]), _ACTIONS, step)\n        return _align_hands(_impact_slots(obs, action), obs)\n    except Exception:\n        farm = _farm(obs, _seat(obs))\n        return {\n            "farmer": ["PASS"],\n            "hands": [["PASS"] for _ in (_get(farm, "hands", []) or [])],\n            "market": [],\n        }\n\n\ndef _kaggle_submission_entrypoint(obs):\n    return agent(obs)\n\n\'\'\'\n\nPREMIUM_SOURCE = r\'\'\'SASH_ACTIONS_B85 = \'c-rk<O>Z1oa{Mnm^B^`!t;RR5)awyeGZG}t5^I4NEZ{W^80*8>H^cwk(vaQNRT&u(neVkmyYOjxTI{O#{W2pXBR~Dm#lQXKm%sh>my3V;bn*M2UcY(u^SiqbAAfqkzj(O3`1im3=fD2f=YRS9@o#_m<v;%V-=9B!`LjR&eD~w)AMV~<Twc6<dw+3x{cySY`os78{kx0HtHVEh*zaF`{`!ago3}q+T>ftJ_5JtzyN{p$`q|<8ckkc6`swAzlYjd4C*QyRwOxk~5C404+Wa5izW@0BX|q3F-0wep{PhP<|F-Uk`;^01pDy0Le)-3r-W|GqarNt$k1y#xdOPHwU-9Pd<^JgnCoPBX**yRAPk&s-Ea}2=NRA(3r^q|*?>_F|t3JeU=1j!#Df_!a+c#Z~<9F=c(@GNkJ3R1lrLNu%-UXh0xs1`Liw|#q+OC|Zolzg=<*CbLij@fld;6Zp5rr$|4_`JX9%MG1^#MKn)2EA<cZc<I>};6Nr~iK($A@?{#k1p~GPui>p<#X;lhT03eYRdLaTJcfcpQ#S7s;qET|G`PI1}zZd?+u_b+fbGx&0=*$)3hI)XS<#cIL68%U`OVlA&DI$}({IUg?Z!eB3f=r&~l0YGLe_>HGC5Vzef#$m4tBhm!-eGkW=h8)NqLTkrXm1y*_P=EHCB*zL}`;g;gY#M5Tn(*ifGIBhf}N5RXtZ{F-*e*F0l`}ZGTzj^&HUuKIuIJWAzO<Jv54$IMY91nzeTWo%dUbX5G`uAr4j-w-61@<koc{^r%VF8(jm$b!+9k4u4KZUjCXaqlf_1m^)`@m*v)*nVM%ai7HS`|C@bRX5)bz&wI+H?37d{E(D1kX9HoNwzfi&v+f^7NNu52@T_n=9)}pZ9OR039c1<>k%TRfdp~s}vErc{tnE=bCh+d&76p@m5O*O1BSPOZbW_I9h$P#W~f~uz+Xwoy4a*ntJ5H2`-vB+XJtA2_pb&>_{KF_s-ay))fOT(BE)$kY7CFkrtdO+M2O@3c`ASU$^g+d-EUi^8NqI)V=8ZTb14&{c+hk-ZMrBVZC!LkCPi?gR~jmt><~2$31O6E_DpMo@RVZ@T*2!ctpW^IKfwF;gEk!{LuLx>GVr>_BA<p)lGif=H~nsp7!ef+jsWU;GG<U0JCrX>#s%v$f9>F&{Q)pP~^21yas!IL5FQ{w`>uhZ5(2O$@>w&J8UpzrrO<#jWESrh$r)0vw=lx^)2=N-TOa<Yi0Q|pA~j148&){syJ=S@pB4_efaqP?*8}t_wWB4sYR_|zjLpmx9lF|>3P~S6n0)?)yA20_{GT}NCWe<5rNoX4)fS}MoivDKK15Ujd5<$SW&33SYylKV)a~xkwv;ZejD*Atk>-9W_*>Dy|KW9ON)k>BXsUqSf>Fi(LDhkK8!l$<G9+&rb%h-6%*PUvbAZSftlQ}Nw?il2IAIIJcLS~xtI|+1q6{O-SKON>{ZT?-83O3;R>^`ZY_3z97SUs3C%=UrxgUoW3#cIx|Vq|KxZN)26L$a#6&J`IO@8ad@wcym~0^S;Ui-4E}{;>9Fta3fB`45$+OP+EG6F-cxzxLIw&Ba_$B)W-3}YR?<{;B4CrxNAz+Nqs@zvnyex9rkIi}fW>P1QC{7+uyFC1r1at!20v$}i-kb#HfhRAQ;*<x+*%*P);$s_(%QHPToLtWvSlrWZuM-37YLpSvM(1|FJ?ONX9K1Ud!bHbt==_U=1t3mMcYtS%!ZqzF-c``d&Y%=f)8lqV;YfM?KArR;s#zGX!ybS8_Va(9yIVZ+&VeoA(}UyVmcB6Cc95<FK$lK76+mn_<k}1nZbRV{XjapMm;e))6=$8qdTQ~O%s8w4Kr_lZixUOrr;%EMS~x%kdd_o7KKmK3E&`&m04-MN@K#?}$(3H^?gCi-21hS$aO=zYu`C<(e8e0?PHi!RV1L`x*!f+L#Mx-1CBVQ5l$LRAr$Bc1B=*G9YldlAU?M`k_6GUZJy)|I!Jp%lpr}bds!8VFfi7S?fSDNcEqky(Vz6a1LH>Xlm^yeZG<fIm#b9Jd1z<XMMBZ{0&YGGAX0P}mN?<aATY#q9DNa=@yoGslhSB2nge9VbOy(r@*s+x2u?vtjI%M({@qhzQsnOsD0Mj@hO#5RBQ82!#GmM0tw?SP^_zoJel88bAlZi%8AlZ;Os}8}^a4WQtJXZE)m62B=y<8W9vQpkd*@4TlkU^@l9{?`FqqP#j6P6~>VyeYgVA{;-x6KIiu1c5EWS-?{i~Tt6T)3>B*8UH#-~90la8`j`q8+<^Rc=^jaii^y-(}P}uz#~YkSp)lV8vGev6gil-C7QthPpfpK_^4+z0}@TW)|V6o&Ic;KR{^<aaBp;r{+9ZG4(E`FhoaJtkrd-6fUKqoRu7ZcbtdqxYvx})hZ18fw~rP6grr7<^o2u84Va{^4cSxzeXTDDrU<G>5S`5QDTMA1A`47r|KlT8M7i$`|JA(ATjK=wetJAwA!TmOzV!R`K7EMHA=t%*~G+FFNADzKzQopa7DkwQU0@*hBZqx3+&h?d0|ETu>$ANsLRp_z!(a48q3EvL#VlsCumf>1fyR$kRxtWTfs4Y+xH(!Szmx6bJ(G$zgx#|xHGw9UyCThmK@`^)_S|!sM-_Ak|6Fcu#dKG<mp$NtDs331#j@p)zgqzW|0JQ`+7iIhDpv~^KG%?dgRCJ3i<IqzgObLZD*`os01)!scY*bYK<|i*&+^Js)+MQgOa*zuU`MN<#EC-kUK$X)TWRi#8@XX#^T%K*4kGVV9v-nK3A?u>T`AU4FS(<(;Axmm2B7&7|qU7613XP8Zi_{U0)fzJjEOF*vUsU#ti@sadi+_qM%t~2FO_akB8f9^ujn66cM2NXjYI{It3sm1~wnim9M2z#j!a~_<C^ygWQe1gHE(F5{b?-YFO%YG+Y}(9b?U+V9&;JZG{U$yGU@3k5TleCAs#-j+j+Mok5TmtaR2$bL_%q*;$&<RyzM#1CV|u!8D(-YY*057DF#9KlrNS1r`I1r<Xa&z8r{eL>oqB1)Na?MVK>&X%nLw4c)eDgML*Sn`p~KuH)2%wFrX(H`D_IJ1v6QCcsos!xd+9fZPYrNtz96j~JS{vc?ZO8`X>BEYHf(rTIISyaYpe^(ju_Y=GUTv|JI#OD0Mv(Iie2^hk3UPme-OM#|%(myJv%A<Dkc0v@^e&lSn3YbDZH8L|_VAXb{wl%+vb0vXVXQ9dHw=L3ftL6vg?tNi5YY|42BE0X{)_vo4jtfHEAMN_xwnTDJs%2YbpB`ZEnp4gl3r(KDff&<UYu*}2#UE4a<RZ3RY{1%|8Tc%&w961(5m{NAiHi1y_+=UuTo0H{u0v<bf{E$e%tyk&Q&Q3+GI;QK$=8i1#=My5EgURh8$*UPt^dCm&=tVL|FTfE~*(kDqP=bzEgxnQPEK-9B*h(>^lr&(f+t*4565JT|@9GazEW+sEt<gb7FM{+zWs8w4b$1>f!+2J<Sa29wB3&Vfp_tHqgpon#X~47w|5YNBUivy9Ci{u1kl`pD)hL|cS;Qd3!lX4732h@`)Fp^|!)$wNGQywnAv!(=8&MFw20mO8$Vu6^{fH-$$q<7|18?kz=#L*m3exbn5tB5MbCu|&l_japd>RIi@QvO-9FXz7@3@<#4?ngxdFHs<_&kG?Y7knp4M=o(A!S860f?U!5$#6r3z1$XHJ<COR^^ny>B|RzHD-(kO{TGsz^!H+`mjv`GfjqbQ_a;lr@`EEb1il;4QwMoF*@^HX2MBk^+*h$fwB+=#QmcPqQn!t;>S;C*e$^muBXSp2hjyI$1`clRW0Y!h4LNHcrMA~gd?U9^;d2@feCwcoq<6gKM)ENi2U0kaMgMeCs{l?&q(N_EVCoXxTk+Tzx;J<PNYuzGH-x(-B9MB3Mv+dxlvJy2tJ8)O@SN9us$>6W)xFRE2w3qN}?_9>*h4Hq-d~M`O__~r4JvnggMSlCz)HtiMG-k(rh^u>>MnSMs6K&vV&=JHPo8+Hg`I7fuKAoyE1U`a9MfDk;cn|+Nv1NM8Pp87vnIV+ogGSTAB<<&OD`+i1I`{GFjZ7R?cYw%phBurhE?9k|H(oPAJGY8Nrk3qL8N!)=Lsjbc4)P6Y8X@^>9wMGsqiv84+-*N-Pn_tK_tSoMJYTKU^9&s?f`9q=rsxKl6aeUDO3;YmWQ%nNyr+5dktsbaHyJPX<^tdBC!L9=&*jd{UqSu>50|AKvhUvIOU0PC86#alG={+hij{$b8J`jOMxW{ir|VU`l>En;Zj|`b~+^8WW1wH=70!#AFmf)()lq#WeXD`e2kzl>w#FDW6QRS3E+RJws>|%V+UJli<eInZ(-CDX1Gnqbq9w4J3@fP)vE+Pl`ntp3jm{?jm15_g)oKWP=qUvEU-|5}4aEEq3PdpyCb++{;{F<u?+iwh8Gl!UHo{6XCSVOvcNBy^z8WWjA`X3=HQ6$f&*J-j%!(HgePh;U1lT8L+{kq4^R$zUkz~U@CeA2>a5O%u66=?PE9-SnUg(lTY2(X=dD$AY;3J##U>V%nCziRyg$Bz7>y`mUc^g4YW2&Uabl*vusb4jMh-qh(0WgeSz0vuzi+_6$|Mf8xwO$GPGO9s;Rb{;Pzg(%hx~^;!Hf;W!d!^pjOsDurnChy0$n%&B>>s*Wu~N{Fd^cic?JmOUumg;u^s^<qbw_Ou^#CB`rL!kVi{xe?Op2^Y-=*;kPG7Oag6;N)CXTVrs3Qn&<Na6a_tzW9SIQ#4U>2iwGCoOEqN&hV(rk$O<xaS&j@nJgnS|G_Tz#TqO<{QGzL+rSKNA<=cq;pPi0(M0^w)9uAo!anJ`;!Fj%vr!miR4=NmiW|D*_E`kR=2XSgD^sJhZ4-dIvPEEp#0~eh&hR_C#cnJ|p+WOX0>{SaNkx-CW%$z1NCpmCfJiff1+V+r<;$92MON+1dnkKL(E6yLq%e!@_x)Jv20K;ONuIsJ-<3-n3w)ZnC-ErXEbV|53g6o~~u9kD}>u`0G-L=t8g8z1cJX4YNOQV<K{=!Lw2oC@#!MbQ<^IsQ=gxZ;#1q;B=f0$4VnTP>{v_s$r>4=6h?FlJ`JJUJr(GvS}7pu)ip&4nM;oy0!WL>JZxju)kpVArGR6<U99<87e!`{pl#;e!15|hLX5b)BftTQ4==oUWq=m0I+KS?fOi=k6=i0WE!ioE$rpOLW*l*r&J+YXHax6K*tBhm&~Iq7CXy_*hT!Yn7epc1bk5O<Jo#a8Otwuy4=Cf1!8(InD~1}nKXjt&?NvYX!7mam*mxg~pu;vAuXaT>~=8aekCfX8(|8)y6+9<3=urd&>13+pSmv8_-#mn*y}smHNI&l*Sk^h_y;!B}bDAdIhIPO29pLLmcQB{+5Az@)74f$B*zog28;Jx)A2K2kNF8ml%7ICQni++5MoB?{0Vq?%35+;LK^qHh-><E}yOx@gxrdc`UTv47*XGn7#E<Hk(a$Edxuue)v}D9i?i1<f!|!pOx}933y6*ZT^MT*tj+VNtWZZZe5^_2Tgib9~$3!~tLbxIc=t!h)c*;@EQpbSvRT&t>N2hw|dR6_8BiL`Q!Ojh2`FT=wxzBk@}M7)dz<;Ef%kkCWusEb6zx&{wO$=35WDj<F1}n~?2M2iiH%!F!C9YT!kh2~1jsmrOLMH5N86V6f8$MKv=R8Qt~w40E|@kQ|*0`*Y`VpZewNJ7FEzK9jc&7%ropHE>Hu3GUlRDwVbrETcjtBm8$&@QmNplN=#R@$%r7l2V#AKUJ+ZM??<zz@}3*U}j5q7WTqV6m*LXO`^+l<?v!Jfiant92=s$5|L0!64qVG3-pXeZ;h}h98($(C8X`O%~(l6q{5~Oj<O}y>G+OPAa)hu5_ACPQJEMEp$KlCRjUF!?HYytX=vu+Lejg({SCL)GL(?1UMuUs7|GRvkGfS!NbTOyr;c|N^(dJ<U7;v9L95n_c>RBJb*O6e_ynDi;Dzyi#$us*3q>V^0bYeNBY?w1@i31h0_t!mk0l~>o1)87*pRyY$&uklIuREZ%h^RAAFYoALc-zojnCMm-e)@qU=P%{Pj)eBq>Vj!fC`%g=2aIt=y<U>-K;9LlX~PuH99(la`m7R1KoiomDSJIv~<UC-2f1p9cY-X);^7~|6;aOz~2}7L9-wpZKtc}{W^{Xjz3ESW0?R2!_^WZVDCbE(J8xLDOR_erd$9$(Imm!cJHdnbYH8oAWQQtW^4=4y-(PWncJo+ILoi6%FeT4F)DSfsjQxEx67O-y7r{fA(f(z9v^~YYKE>D04X98Gw4hppa!vADoIALo--H;8uT>tnDJBgRvo2r?iM^R8JHpK3}zaNb4VB<b}Q45xDr3}Q$Yguq&$+uzp@T;rgW8izT>qhDKARKqYYtz61R$Ef@!Tzn&H$c?Lid)QVNclrBNb_b)^MO<w)KZwm$*fLeYKCN2blD8Rx96j2bdVIHJkqloLxkL$1+*8zV9(%=~yttMHQHOeAQhfp3BoPVE;G+Z>Q0h;U}LX#{7T+c2gp5z|`0CFLk3td$2$|GdO&`5Fk!)vi)={Tq;E99C+)F67nvWdJEPCC}}vNnq=^d-R?dG{!U~^#l5uy2tJ1i=j)Dpa%S8RqFvV;=5;*7Sffl@YJ0%Z+nu=f#s*uBoU@)*pk4Aa91A&G_x*vty>RR1NLDJ4;=YRF!=f$Vuigr`jobdTL{>b3ihms2FnE3j&4o2U3jaGzc+zA|J^7H<J7lPf^b>x$yikx=vpYppT#jc`ApQmrtyA1$rJa9NuEtnX3Yq1aIF3O%N+Er4YM~Yu3T)Zs0Gz{V3}X_0;PjX(mOWrkgA1P$+@EyQ#~J+GP+v1|7^f$VhEG;=YQWQrU#m|uhbZDkc3i$A_#Nd20^#g4v0I|38`o(!<<r_!4U8G_Q)7lyRVSWV*L(*x@{~VmwI{p#ci^bZkySo3e@lt?NR*DOgSP14ah;@K?e$fbc~O9d*YiW`>gB>=NW$V>?Y6kVqvNx8ex}RQ`Vz2Dm!Qn&#;I>hb;6@sM<s2Ogy4H8&RZoK|{E+Wa3Dw3S+52Kd7b1KHhZ6bK*K;M5NCK>UNd(PT8K+=MX6#XA0kwy%_f!AxUh3;nLwix$T0UG;#s-ccpMMsVY7<_$kLj`@fz*gk4qFEV*6ab3$mA3f($oYLk<%036WtG?CILv35o58B)owsHb3F8&T0ALJ=K|gZku@Y0+sC1EN9bmMOjhNyEtTBcc?UE6TY6x5l}nQdAfdB>oY7NK?eje;>%V029V49eSLbxK_dZvL=A+3)Wo}{TU$zu)G>n86$RXUI+5B0_R{j<Psgv&#tVQ2gc8wZ!-P|11To!>>HYBa##wLDEG-?Q+;cwfzZ+BrIX6#D40{?$$*&fONA4{>dD_b#A#-cS94^FnGcT%qZMICA(0?VbXzlIlp^9}d9o59!iww3nWp6@I-~_b=ka;@T<ye932HDgt|~wttLumkI{|gDT>2S#PD%nL^BEE;Lvapls-{}<0F7Sm9tJdvX)YJzhUA0Alpi<^t%hX6bAeSOXbX4%g@j(t19%xeoP{!vE6^+b7cO_966Gt%YL@pvF&AAak%Xp2+8?Zhv}8RdJRE~aQCk|6NT&f9V00<A(I{$_@?Hvn3@z7Q)zoS6J6si8HLlE2?|{OwoTIoi{UXOaE$G;4M>AW%nfy(R1`#a)Mv-VO2Hv;`eNlM}cC_;mO)#<+sg#WoKtTX$16DaD!_0K7GC@_95Y*~A8}~Q{=@P=&5W#;YG(FMiiOnYpCqUT;sTG+_4@U}jS7hXT4Ru)wT{Trw&d1W>_zzFgS#gnGgbj8}G08#{xIo%r`XkMpCLf(*ektmMP&uHsi7)e7R*H0a11v<$I1)8?HTk;I(L(<?LBgL0QzJ!J<!#5qy^2FdQUkCtXep#N|AT?M&HA~P)PbB#k!9ZQ9o$SH0*_~i=&nTL70pUj8QrTc#Hayo9VC?l)kk^>`Lf`wU{6Sr0{5l{imh*OMi|a4X#r$L3<8Y`L=2QJMtg=5cpJclQ>nL~YwO<eMiLd&hr{rP8<v}(w+@_auVOT`moPq23jO)1=$=e(226o`YR+Kd1z!2m6dQ~uM$>FLRC=@$a#Ua|g%ivUR@h}i!)Gt0kpz@P^Tj&Ia>Cy6S})6a$vil&JyoKp;ggmppI+K!%vs4Jt5y&Lmx=@@v$#6Q)lx~l*=Vs;w+2&=9)pa0n4N|MCQBkSWbaGh$*)<p)3wyhvx-UUuuKaJN@V1_(>c7IQAx^4+{7!5Z9Plpc~H=2cP6^q_1Q5)Bt2fQG4;(W{bc;Mgv@4_*JfL-C70Yv?V}rhOla7qq@{^+a%g>?P_GL{4%iS;h#^X{{!=A|k*%x|V%M%RNAZmj5Q6h0CXebB_FZ*_gV$5#=4OjkQci36mp1$p=MOVo^4zgb<Ug{l-~v&5g7PS`X)ejD30%}yVpoN*XujaI>OL^`K#>uWB4`$6F2X$7LF<?3)*{|&T^%h-nA0reZ)76Hb4M}V=N$nWG2(3Oebw7$5>VniQm%YLyE7rUvG$r*krLgCgbRX|!DaCnxqe~OM?5Yz+~N)zlEF7g;+TdaJL*XV(x{hFjHHP@A$TwYH{}_?3mEp&*TD1Q{7vNVUXWl-xR~OwTabDRX-vdCUFCP8-kz&>&l_t-3iOa2kKJZ47vfQ8@(O8NSe>i;{UCTK1-Cs-_9_bJX<io=%{L$cwfoK(yRXA8J6~>O4#ESCsBm3Z36cyUp!tlEkdcB)wLKmQ9ya@}6hdxR+rF-_k<|St{g#A#;mboe#KhHbHp_q4L`^C_l5}JDRnh4gYCTM}+zR*iIG9-ebU&!e1&z9`J5??x1^XljBSZoD*j*5Y0h;chyzpMdM6Y(O40MS<S{jwUrgj5-y(*PZudO4oEl>YBfu|$Z2iE4KUb6g4v2kUVs#8j-i6L+{A?I^pk^0#E$6dtj72ls0s&Vn05ox9;&fGekt0aXkUJLvPaNJlja#Rofc;5JmxAgU#0mPL24ug5$Or|0lw~DZ{IvGMbj@n}`65*Z|tRpcIpL(1oW||LuV~jf_Sl5u{M2+F$U9SpOAYn<-S`D3=4}DNrlTULMX`yjLw7XuUj<lJ}T(dT-62}BN=?v+(=Cy}*jAi3}dxf)JR2Vi>^{u}OI+8#k88PbSp&(V%Qi<|V6JJEwuCK+CeONBn{U8vstLO>|Wevw~*!RVEubXL?ivB}O4Q{)6){wJp)HB`%lROy8u>>pQa6Px{v%AKOjipe3H^K+H&QDH9%CD=EYsCwX@kW_-01Fh7S;q*=E3OE5NueTN);>PAL@_E1qN|T$33d#CKU|@RtEI%*qQ@iGk*dWGfT?F=pi<AOdjC#Z{Lr>e!U+J<B*hrpBTra#Jh<&*RLmze)Mrw8A7-SPRnBRG4g;tv+LP|laPN_ZsN}tNeLY>3cqq^eKrgKa;X)ljoEr_>GnFtJaxa5z`4XzT_|Z+EfG{blfUBuP`dqdCv}d_c_>v{OWs52zxf^W78175=#`G=p0IbGR->kR2UY$<I?`(SCmF9=4>U&X+ONieSDX{KxD`!D{__C{(Tl)6w=}&f|&)GzhCgwVcDsGpd=qp})-?c<Kgqo*U?4iyWFb-zm{*@4rcW>@~T;7PksfzcsY*dq78e69_6NE)INQN;|z%sF>&=c3J_S1E!_({n-v*m87MY*UQgwD?Zv@93ma6y89imjqLtri)`Q@aS~Ol(TL(1l7#1fG+<V}jw(hFM=~MU`$rS|(9>(3S2ko2Cck`d1L7h|ndC;p$^|B%3qaoS3HvmTJDm0NRx9{O#JhEFTDPo+9kZom!O@z-Qa!SDA@3)Q)y|%U7i)Qy)M2=B(sp(@?molu~t!8K`y<4+%&k%m1vLbYq;IKmlc6s`xIjnqsXoZ9$L*r}R$y?5DXmY??dmUaIrdRuowdB14>u$4j$<FwVr20o%pBTcr*%B=&R2ElYnniNv=9#|$8dh(5C%gjvq+M$__j_#v!|@ii*U#n(x3#pDMRS;P?sYBiPPq~0NJWPP!Y2-t3MgCf^Rb~###Tqg}BZ^cy!KrrKq^0Lf0m5nVPYFbe0TkOze_%5DA=g~tyM+H9#^ZDXP{b%}`M&HroG(Ta&>R$KX{PKNv=|4E6JulBB^xIX73}V_eh5&^)J+`@-%-3nMJnNhF)ytfuOuD*5Wn-+nsF*@jlQJc+dety*QOC;!Mu;kc?j4!s>j&S(pwh~0QF|+6j*ndtU#etT`q`kfGHXvgl!BNyF&hC;XMo@}vKJ|)C?W$|YDaTlk}=B?bPL&HLR_U=e+{8Gd0j(li_%B+CCd8>$fA&(1*5t}OA;YtB^WN2WRl#dwnefEU4?I{+8kZ1`#H2hTCtFErwwGaQjbFEaCz;__mQgZ&`VE^0o8ja0PkW1luzm_55UxqEbH*z)U6Wgd@Qx9UvjpSqU{yfi^v6+tz4(wm00Xdj3}Zn1U0J`3XF}HNCcH60&ofvA_$?FIM&CX9eJxoeQ2NIwKJo`-+=nh)w61$xn;5Vd0aHYLP=5_C3&2NHT@}aR*P9|zWkA*cY&NYfEV~rsazZMM#cxL>g7GDhQnn`%E5S6{7rQOFbygt3fmGFnX&B^)$kJ`Xae1A9UiZWiR$jzOV!!QOde;^D%mt%EJi8KmLpQL*obUJ5P5Cse$!gF(1CML4toK390E=lC}5gS1crq&)H2qdp5l^QA;iu$wH8sSG6A(76iq3Kyq5@j6pEKsYD_7ap=z-W0X514VlV+epRB-FP3<*Npe_kyDerRBU1y#bh1Dw*G%9gBh9XK*i_q8ITl!J}V$cbaWMth^e=xBVpzT?g6Fy-FA49?t;sA;98<7pX&IUXNTsQ@(Sf(p4`v}LF7G6u2g{(H<$f$z_l!ze#_6Yroz#lD@<~YhxZ#cp@@f%X#BP)}|8Mz{D5(3oZO-s`g_WD+M%Kkz85I{6>6b@i9>-zM!*BSP5ZKrL$sD|BHBCT-3T$5YfW`tyIys%PonA(NJ(jB0%F$uzw@axbNP=AWL-?Xva>9I9~nZSL;*z#vd6e20tWN4LgqxBhO&{k$r6sYP2p=jzId-YVC$#+T=PZin5y(qMb(mP92qr}H4?9*)_cNmoW18wGZ<X<W~6>63{!h|F2<O(N>QWcPhEW3lYmcxrmkz7ws0}9`W6UR8~V#AJor|kk7CR81BYlV}vu~k&)fKfejOD@HPk{B(G*BOYY0+M!Cyr?b8c;tGS5iX(;8ao$-L0RYEK9}9Qwk1LHmano&Lz`pt*w*shE={&$ub&lKT=!u}cfjSWo15;3%D|Jsc0Odgs8p}{cwn|gSV#v4KP<!pL*eHzF;}IYJp3QSe5Pd\'\n\n\nimport base64\nimport copy\nimport json\nimport math\nimport zlib\n\n_ACTIONS = json.loads(zlib.decompress(base64.b85decode(SASH_ACTIONS_B85)).decode())\n\n_MODEL_MEAN = (0.7754330004241974, 0.42216157898077045, 4.845066681290341, 0.2690391571999746, 0.649190802764908, 0.5422947717626129, 0.38474963508282034)\n_MODEL_SCALE = (0.9129241235454895, 0.4939040192317694, 3.256390917597153, 0.44346035797258343, 0.4421957956795104, 0.27905760741549984, 0.4865360761410145)\n_MODEL_INTERCEPT = 1.245742223898873\n_MODEL_COEFFICIENTS = (-0.9451280735752147, -0.4680272533738784, -1.4615973335974357, 0.599421907433265, -0.1545610769891183, 0.44222011734216526, -0.025622914463126638)\n_MODEL_THRESHOLD = 0.8065185529227787\n_SELLABLE = (\n    "STRAWBERRY", "MELON", "MILK", "WOOL", "EGG",\n    "TOMATO", "CARROT", "WHEAT", "FERTILIZER",\n)\n_PRODUCT_BY_ANIMAL = {"COW": "MILK", "SHEEP": "WOOL", "GOOSE": "EGG"}\n_GLUT_WEIGHT = {\n    "STRAWBERRY": 2.0, "MELON": 3.6, "MILK": 2.0, "WOOL": 3.2,\n    "EGG": 1.5, "TOMATO": 1.3, "CARROT": 1.0, "WHEAT": 1.0,\n    "FERTILIZER": 1.0,\n}\n_STATE = {0: {}, 1: {}}\n_WEED_STATE = {0: {}, 1: {}}\n_WEED_REPLAY_STEPS = 8\n\n\ndef _get(value, key, default=None):\n    if isinstance(value, dict):\n        return value.get(key, default)\n    getter = getattr(value, "get", None)\n    if callable(getter):\n        return getter(key, default)\n    return getattr(value, key, default)\n\n\ndef _copy_action(action):\n    action = copy.deepcopy(action or {})\n    return {\n        "farmer": list(action.get("farmer") or ["PASS"]),\n        "hands": [list(order or ["PASS"]) for order in (action.get("hands") or [])],\n        "market": [list(order) for order in (action.get("market") or [])],\n    }\n\n\ndef _tile_token(tile):\n    if tile is None or isinstance(tile, str):\n        return tile\n    return tuple(\n        (key, _get(tile, key))\n        for key in (\n            "kind", "crop", "animal", "growth", "yield_units",\n            "fertilizer_available", "cared_today",\n        )\n        if _get(tile, key) is not None\n    )\n\n\ndef _public_farm_distance(left, right):\n    distance = 0\n    if tuple(_get(left, "farmer", []) or []) != tuple(_get(right, "farmer", []) or []):\n        distance += 2\n    left_hands = [tuple(position or ()) for position in (_get(left, "hands", []) or [])]\n    right_hands = [tuple(position or ()) for position in (_get(right, "hands", []) or [])]\n    distance += 3 * abs(len(left_hands) - len(right_hands))\n    distance += sum(a != b for a, b in zip(left_hands, right_hands))\n    left_unlocks = set(_get(left, "unlocked_quadrants", []) or [])\n    right_unlocks = set(_get(right, "unlocked_quadrants", []) or [])\n    distance += 4 * len(left_unlocks.symmetric_difference(right_unlocks))\n    left_tiles = _get(left, "tiles", []) or []\n    right_tiles = _get(right, "tiles", []) or []\n    for y in range(max(len(left_tiles), len(right_tiles))):\n        left_row = left_tiles[y] if y < len(left_tiles) else []\n        right_row = right_tiles[y] if y < len(right_tiles) else []\n        for x in range(max(len(left_row), len(right_row))):\n            a = _tile_token(left_row[x]) if x < len(left_row) else "MISSING"\n            b = _tile_token(right_row[x]) if x < len(right_row) else "MISSING"\n            distance += a != b\n    return distance\n\n\ndef _mirror_probability(distance, money_gap, board_streak, step):\n    values = (\n        math.log1p(max(0.0, distance)),\n        float(distance == 0),\n        math.log1p(max(0.0, money_gap)),\n        float(money_gap <= 5.0),\n        min(max(0, board_streak), 96) / 96.0,\n        min(max(0, step), 718) / 718.0,\n        float(step >= 480),\n    )\n    logit = _MODEL_INTERCEPT + sum(\n        coefficient * ((value - mean) / scale)\n        for value, mean, scale, coefficient in zip(\n            values, _MODEL_MEAN, _MODEL_SCALE, _MODEL_COEFFICIENTS\n        )\n    )\n    return 1.0 / (1.0 + math.exp(-min(35.0, max(-35.0, logit))))\n\n\ndef _align_hands(action, obs):\n    action = _copy_action(action)\n    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0\n    farms = list(_get(obs, "farms", []) or [])\n    farm = farms[seat] if seat < len(farms) else {}\n    expected = len(_get(farm, "hands", []) or [])\n    hands = list(action.get("hands") or [])\n    if len(hands) < expected:\n        hands.extend([["PASS"] for _ in range(expected - len(hands))])\n    action["hands"] = [list(order or ["PASS"]) for order in hands[:expected]]\n    return action\n\n\ndef _tile_at(farm, position):\n    try:\n        x, y = int(position[0]), int(position[1])\n        return (_get(farm, "tiles", []) or [])[y][x]\n    except (IndexError, TypeError, ValueError):\n        return "LOCKED"\n\n\ndef _trace_actor_action(step, actor):\n    trace = _ACTIONS[min(max(int(step), 0), len(_ACTIONS) - 1)] or {}\n    if actor == "farmer":\n        return list(trace.get("farmer") or ["PASS"])\n    hands = trace.get("hands", []) or []\n    return list(hands[actor] if actor < len(hands) else ["PASS"])\n\n\ndef _weed_repair_action(obs, action, step):\n    """Repair a visible fixed-route PLANT/BUILD slip without touching market."""\n    action = _align_hands(action, obs)\n    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0\n    game = _WEED_STATE[seat]\n    if step == 0 or step < game.get("last_step", -1):\n        game = {"last_step": step, "active": {}}\n        _WEED_STATE[seat] = game\n    game["last_step"] = step\n\n    farms = list(_get(obs, "farms", []) or [])\n    farm = farms[seat] if seat < len(farms) else {}\n    positions = [_get(farm, "farmer"), *list(_get(farm, "hands", []) or [])]\n    unit_actions = [action.get("farmer", ["PASS"]), *list(action.get("hands") or [])]\n    active = game["active"]\n\n    for actor, transaction in list(active.items()):\n        index = 0 if actor == "farmer" else int(actor) + 1\n        if index >= len(unit_actions):\n            active.pop(actor, None)\n            continue\n        age = step - transaction["start"]\n        if age == 1:\n            unit_actions[index] = list(transaction["intended"])\n        elif 2 <= age <= 1 + _WEED_REPLAY_STEPS:\n            unit_actions[index] = _trace_actor_action(step - 1, actor)\n        else:\n            active.pop(actor, None)\n\n    for index, (position, intended) in enumerate(zip(positions, unit_actions)):\n        actor = "farmer" if index == 0 else index - 1\n        if actor in active or not isinstance(intended, list) or not intended:\n            continue\n        operation = intended[0]\n        if operation not in ("BUILD_PASTURE", "PLANT"):\n            continue\n        tile = _tile_at(farm, position)\n        if not isinstance(tile, dict) or tile.get("kind") != "WEED":\n            continue\n        active[actor] = {"start": step, "intended": list(intended)}\n        unit_actions[index] = ["DIG"]\n\n    action["farmer"] = unit_actions[0] if unit_actions else ["PASS"]\n    action["hands"] = unit_actions[1:]\n    return _align_hands(action, obs)\n\n\ndef _shed_access(size):\n    half = size // 2\n    return {\n        (half - 1, half - 1), (half, half - 1),\n        (half - 1, half), (half, half),\n    }\n\n\ndef _projected_shed(obs, action):\n    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0\n    farms = list(_get(obs, "farms", []) or [])\n    farm = farms[seat] if seat < len(farms) else {}\n    private = _get(obs, "private", {}) or {}\n    projected = {\n        key: max(0, int(value or 0))\n        for key, value in dict(_get(private, "shed", {}) or {}).items()\n    }\n    inventories = list(_get(private, "inventories", []) or [])\n    positions = [_get(farm, "farmer", [0, 0]), *list(_get(farm, "hands", []) or [])]\n    actions = [action.get("farmer", ["PASS"]), *list(action.get("hands") or [])]\n    tiles = list(_get(farm, "tiles", []) or [])\n    access = _shed_access(len(tiles) or 10)\n    for index, unit_action in enumerate(actions):\n        if index >= len(positions) or index >= len(inventories):\n            continue\n        position = positions[index]\n        if not isinstance(position, (list, tuple)) or len(position) < 2:\n            continue\n        x, y = int(position[0]), int(position[1])\n        if (x, y) not in access or not (0 <= y < len(tiles) and 0 <= x < len(tiles[y])):\n            continue\n        if tiles[y][x] == "LOCKED" or not isinstance(unit_action, list) or not unit_action:\n            continue\n        inventory = {\n            key: max(0, int(value or 0))\n            for key, value in dict(inventories[index] or {}).items()\n        }\n        if unit_action[0] == "DROP":\n            deposits = inventory.items()\n        elif unit_action[0] == "PLACE" and len(unit_action) >= 2:\n            item = unit_action[1]\n            tile = tiles[y][x]\n            structure = {"COW": "PASTURE", "SHEEP": "PASTURE", "GOOSE": "COOP"}.get(item)\n            if (\n                structure is not None and isinstance(tile, dict)\n                and tile.get("kind") == structure and "animal" not in tile\n            ):\n                continue\n            try:\n                requested = int(unit_action[2]) if len(unit_action) >= 3 else 1\n            except (TypeError, ValueError):\n                continue\n            deposits = ((item, min(max(0, requested), inventory.get(item, 0))),)\n        else:\n            continue\n        for item, quantity in deposits:\n            room = max(0, 100 - sum(projected.values()))\n            amount = min(max(0, int(quantity or 0)), room)\n            if amount:\n                projected[item] = projected.get(item, 0) + amount\n    return projected\n\n\ndef _opponent_exposure(obs):\n    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0\n    farms = list(_get(obs, "farms", []) or [])\n    opponent = farms[1 - seat] if len(farms) >= 2 else {}\n    exposure = {item: 0.0 for item in _SELLABLE}\n    for row in (_get(opponent, "tiles", []) or []):\n        for tile in row if isinstance(row, list) else [row]:\n            if not isinstance(tile, dict):\n                continue\n            crop = str(tile.get("crop", "")).upper()\n            if crop in exposure:\n                exposure[crop] += max(1.0, float(tile.get("yield_units", 0) or 0))\n            product = _PRODUCT_BY_ANIMAL.get(str(tile.get("animal", "")).upper())\n            if product:\n                exposure[product] += 1.0 + max(0.0, float(tile.get("yield_units", 0) or 0))\n            if tile.get("fertilizer_available", False):\n                exposure["FERTILIZER"] += 1.0\n    return exposure\n\n\ndef _ranked_sells(obs, action, requested=None):\n    shed = _projected_shed(obs, action)\n    if requested is None:\n        requested = {item: int(shed.get(item, 0) or 0) for item in _SELLABLE}\n    prices = _get(_get(obs, "market", {}) or {}, "prices", {}) or {}\n    exposure = _opponent_exposure(obs)\n    rows = []\n    for index, item in enumerate(_SELLABLE):\n        quantity = min(max(0, int(requested.get(item, 0) or 0)), max(0, int(shed.get(item, 0) or 0)))\n        if quantity <= 0:\n            continue\n        score = (\n            (1.0 + exposure.get(item, 0.0))\n            * _GLUT_WEIGHT.get(item, 1.0)\n            * max(1.0, float(prices.get(item, 1) or 1))\n            * math.log1p(quantity)\n        )\n        rows.append((score, -index, item, quantity))\n    rows.sort(reverse=True)\n    return [["SELL", item, quantity] for _, _, item, quantity in rows]\n\n\ndef _front_run_market(obs, action):\n    action = _align_hands(action, obs)\n    market = [list(order) for order in (action.get("market") or [])]\n    requests = {}\n    for order in market:\n        if len(order) >= 3 and order[0] == "SELL":\n            try:\n                quantity = max(0, int(order[2]))\n            except (TypeError, ValueError):\n                quantity = 0\n            requests[order[1]] = requests.get(order[1], 0) + quantity\n    sells = _ranked_sells(obs, action, requests)\n    targeted = {order[1] for order in sells}\n    remainder = [\n        order for order in market\n        if not (len(order) >= 3 and order[0] == "SELL" and order[1] in targeted)\n    ]\n    action["market"] = (sells + remainder)[:10]\n    return action\n\n\n_TERMINAL_PRODUCTS = set(_SELLABLE)\n_TERMINAL_TIE_PRIORITY = {"WOOL": 8, "MELON": 7, "MILK": 6, "STRAWBERRY": 5, "CARROT": 4, "FERTILIZER": 3, "WHEAT": 2, "EGG": 1}\n\ndef _best_terminal_item(inventory, prices):\n    choices = []\n    for item, quantity in dict(inventory or {}).items():\n        quantity = int(quantity or 0)\n        if item not in _SELLABLE or quantity <= 0:\n            continue\n        price = int(prices.get(item, 0) or 0)\n        choices.append((price * quantity, price, quantity, _TERMINAL_TIE_PRIORITY.get(item, 0), item))\n    return max(choices, default=None)\n\n\ndef _terminal_overlay(obs):\n    private = _get(obs, "private", {}) or {}\n    inventories = list(_get(private, "inventories", []) or [])\n    shed = dict(_get(private, "shed", {}) or {})\n    prices = _get(_get(obs, "market", {}) or {}, "prices", {}) or {}\n    player = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0\n    farms = list(_get(obs, "farms", []) or [])\n    farm = farms[player] if player < len(farms) else {}\n    hand_count = len(_get(farm, "hands", []) or [])\n    placed = {}\n    farmer = ["PASS"]\n    if inventories:\n        choice = _best_terminal_item(inventories[0], prices)\n        if choice is not None:\n            _, _, quantity, _, item = choice\n            farmer = ["PLACE", item, quantity]\n            placed[item] = quantity\n    hands = []\n    for index in range(hand_count):\n        inventory = inventories[index + 1] if index + 1 < len(inventories) else {}\n        choice = _best_terminal_item(inventory, prices)\n        if choice is None:\n            hands.append(["PASS"])\n            continue\n        _, _, quantity, _, item = choice\n        hands.append(["PLACE", item, quantity])\n        placed[item] = placed.get(item, 0) + quantity\n    totals = {item: int(shed.get(item, 0) or 0) + int(placed.get(item, 0) or 0) for item in _TERMINAL_PRODUCTS if int(shed.get(item, 0) or 0) + int(placed.get(item, 0) or 0) > 0}\n    ordered = sorted(totals, key=lambda item: (int(prices.get(item, 0) or 0) * totals[item], int(prices.get(item, 0) or 0), totals[item], _TERMINAL_TIE_PRIORITY.get(item, 0)), reverse=True)\n    return {"farmer": farmer, "hands": hands, "market": [["SELL", item, totals[item]] for item in ordered[:10]]}\n\n\ndef _hybrid_action(obs, action, step):\n    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0\n    game = _STATE[seat]\n    if step == 0 or step < game.get("last_step", -1):\n        game = {\n            "board_streak": 0, "latched": False, "divergence": 0,\n            "last_step": step, "mode": None,\n        }\n        _STATE[seat] = game\n    game["last_step"] = step\n    farms = list(_get(obs, "farms", []) or [])\n    distance = _public_farm_distance(farms[0], farms[1]) if len(farms) >= 2 else math.inf\n    money_gap = (\n        abs(float(_get(farms[0], "money", 0) or 0) - float(_get(farms[1], "money", 0) or 0))\n        if len(farms) >= 2 else math.inf\n    )\n    game["board_streak"] = game["board_streak"] + 1 if distance <= 2 else 0\n    probability = _mirror_probability(distance, money_gap, game["board_streak"], step)\n    selected = probability >= _MODEL_THRESHOLD\n    if selected and not game["latched"]:\n        game["latched"] = True\n    if game["latched"]:\n        game["divergence"] = 0 if distance <= 2 else game["divergence"] + 1\n        if game["divergence"] >= 8:\n            game["latched"] = False\n            game["divergence"] = 0\n    if game["mode"] is None and step >= 48:\n        game["mode"] = "mirror" if selected else "open"\n    active = step >= 24 and (\n        game["latched"] if game["mode"] == "mirror" else game["mode"] == "open"\n    )\n    if active:\n        action = _front_run_market(obs, action)\n    if step >= 716:\n        return _terminal_overlay(obs)\n    return _align_hands(action, obs)\n\n\ndef agent(obs):\n    """Kaggle one-argument entry point."""\n    try:\n        step = min(max(0, int(_get(obs, "step", 0) or 0)), len(_ACTIONS) - 1)\n        action = _weed_repair_action(obs, _copy_action(_ACTIONS[step]), step)\n        return _hybrid_action(obs, action, step)\n    except Exception:\n        seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0\n        farms = list(_get(obs, "farms", []) or [])\n        farm = farms[seat] if seat < len(farms) else {}\n        return {\n            "farmer": ["PASS"],\n            "hands": [["PASS"] for _ in (_get(farm, "hands", []) or [])],\n            "market": [],\n        }\n\n\ndef _kaggle_submission_entrypoint(obs):\n    return agent(obs)\n\'\'\'\n\nimport copy\n\n_V22_NS = {}\n_PREMIUM_NS = {}\nexec(V22_SOURCE, _V22_NS)\nexec(PREMIUM_SOURCE, _PREMIUM_NS)\n_ROUTE = {0: None, 1: None}\n\n\ndef _get(value, key, default=None):\n    if isinstance(value, dict):\n        return value.get(key, default)\n    getter = getattr(value, "get", None)\n    if callable(getter):\n        return getter(key, default)\n    return getattr(value, key, default)\n\n\ndef _choose_route(obs):\n    prices = _get(_get(obs, "market", {}) or {}, "prices", {}) or {}\n    melon = float(_get(prices, "MELON", 0) or 0)\n    strawberry = float(_get(prices, "STRAWBERRY", 0) or 0)\n    farms = list(_get(obs, "farms", []) or [])\n    if len(farms) != 2:\n        return "v22_base"\n    if melon >= 200 or strawberry >= 150:\n        return "premium_control"\n    return "v22_base"\n\n\ndef agent(obs):\n    seat = 1 if int(_get(obs, "player", 0) or 0) == 1 else 0\n    step = int(_get(obs, "step", 0) or 0)\n    if step == 0 or _ROUTE[seat] not in ("v22_base", "premium_control"):\n        _ROUTE[seat] = _choose_route(obs)\n    if _ROUTE[seat] == "premium_control":\n        return _PREMIUM_NS["agent"](obs)\n    return _V22_NS["agent"](obs)\n\n\ndef _kaggle_submission_entrypoint(obs):\n    return agent(obs)\n'

## Safety contract

Both routes are deterministic traces. The selector only chooses between complete agents at step zero. Farmer, hand, and market actions are produced by the selected source without hybrid mutation.


In [ ]:
from pathlib import Path
import hashlib
compile(AGENT_SOURCE, "main.py", "exec")
Path("main.py").write_text(AGENT_SOURCE)
print(f"routes: v22_base + premium_control")
print(f"main.py ready: {len(AGENT_SOURCE):,} characters")
print(f"sha256: {hashlib.sha256(AGENT_SOURCE.encode()).hexdigest()}")
